# J9 — MBEC Manuscript Results Pack

Single notebook that regenerates **every number, table and figure** required by the
`CalStack-SSL` MBEC manuscript, and emits ready-to-paste LaTeX table bodies.

## Required Kaggle inputs (Add Input → Notebook Output)
| Attach | Provides |
|---|---|
| `last-part-of-the-previous-one` | pipeline root: 7 frozen sources, SimCLR/Rotation/fusion weights |
| `confidence-gated-ensemble` | `final_ensemble_config.json`, `final_internal_probabilities.npz`, stacker, external + PAD results |
| `supervised-classification-for-journal` | supervised 5-seed probabilities, locked split IDs |
| `journal-extension-j1` | protocol lock, `original_80_10_10_locked_ids.csv` |
| `journal-extension-j2` | DINOv2 same-split probabilities + features |
| `journal-extension-j5` | validation features (`val_features.npz`) for CKA |
| `journal-extension-j6` | (optional) prior calibration outputs |
| `journal-extension-j7` | (optional) Grad-CAM + source-importance figures |
| `j8-final-journal-results-consolidation` | (optional) prior figure pack |
| `output-result-for-pad-fusion` | PAD adapted results |

**Accelerator: GPU T4/P100.** Only Section 8 (efficiency) needs it; everything else runs on CPU.

## Outputs
`/kaggle/working/J9_MBEC_RESULTS_PACKAGE.zip` containing
`tables/` (CSV) · `latex/` (paste-ready `.tex` bodies) · `figures/` (Fig1–Fig8 + ESM) ·
`audit/` · `MANUSCRIPT_VALUES.json` (every `\TBD{}` in the manuscript, keyed).

In [1]:
# ============================================================
# CELL 1 — CONFIG
# ============================================================
import os, gc, json, math, shutil, zipfile, warnings, time, hashlib
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

KAGGLE_INPUT = Path("/kaggle/input")
WORKING      = Path("/kaggle/working")

ROOT      = WORKING / "j9_mbec_results"
TABLE_DIR = ROOT / "tables"
LATEX_DIR = ROOT / "latex"
FIG_DIR   = ROOT / "figures"
ESM_DIR   = ROOT / "figures_esm"
AUDIT_DIR = ROOT / "audit"

for d in [ROOT, TABLE_DIR, LATEX_DIR, FIG_DIR, ESM_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
NUM_CLASSES = 8

EXPECTED_VAL_N  = 2533
EXPECTED_TEST_N = 2534

N_BOOTSTRAP    = 5000
BOOTSTRAP_SEED = 42
ECE_BINS       = 15
COVERAGES      = [1.00, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30]

# Efficiency measurement
LATENCY_WARMUP  = 10
LATENCY_REPEATS = 50

# Master registry of every value the manuscript needs.
MANUSCRIPT = {}

def record(key, value, note=""):
    """Store a value destined for the manuscript, keyed by its \\TBD{} slot."""
    MANUSCRIPT[key] = {"value": value, "note": note}
    return value

print("=" * 100)
print("J9 — MBEC MANUSCRIPT RESULTS PACK")
print("=" * 100)
print("Output root :", ROOT)
print("Bootstrap   :", N_BOOTSTRAP, "replicates, seed", BOOTSTRAP_SEED)

J9 — MBEC MANUSCRIPT RESULTS PACK
Output root : /kaggle/working/j9_mbec_results
Bootstrap   : 5000 replicates, seed 42


In [2]:
# ============================================================
# CELL 2 — HELPERS
# ============================================================
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef,
    classification_report,
)
from sklearn.preprocessing import label_binarize
from scipy.stats import binomtest


def find_all(filename):
    return sorted(KAGGLE_INPUT.rglob(filename), key=lambda p: (len(p.parts), str(p)))


def find_one(filename, prefer=()):
    hits = find_all(filename)
    if not hits:
        return None
    def rank(p):
        s = str(p).lower()
        pref = 0
        for i, t in enumerate(prefer):
            if t.lower() in s:
                pref -= (100 - i)
        return (pref, len(p.parts), str(p))
    return sorted(hits, key=rank)[0]


def normp(probabilities, eps=1e-12):
    p = np.asarray(probabilities, dtype=np.float64)
    if p.ndim != 2 or p.shape[1] != NUM_CLASSES:
        raise ValueError(f"Expected (N,{NUM_CLASSES}); got {p.shape}")
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, eps, None)
    p /= p.sum(axis=1, keepdims=True)
    return p.astype(np.float32)


def probability_columns(df):
    cols = [c for c in df.columns
            if c.startswith("prob_class_") and c.rsplit("_", 1)[-1].isdigit()]
    if len(cols) == NUM_CLASSES:
        return sorted(cols, key=lambda c: int(c.rsplit("_", 1)[-1]))
    cols = [c for c in df.columns
            if c.startswith("prob_") and len(c.split("_")) >= 2 and c.split("_")[1].isdigit()]
    if len(cols) == NUM_CLASSES:
        return sorted(cols, key=lambda c: int(c.split("_")[1]))
    exact = [f"prob_{i}" for i in range(NUM_CLASSES) if f"prob_{i}" in df.columns]
    if len(exact) == NUM_CLASSES:
        return exact
    raise ValueError(f"Could not identify 8 probability columns in {list(df.columns)[:20]}")


def label_column(df):
    for c in ["y_true", "true_label", "label", "y"]:
        if c in df.columns:
            return c
    raise ValueError("No true-label column found.")


def load_prediction_csv(path):
    df = pd.read_csv(path)
    y = df[label_column(df)].to_numpy(np.int32)
    p = normp(df[probability_columns(df)].to_numpy(np.float32))
    return y, p


def temperature_scale(probabilities, T, eps=1e-12):
    logp = np.log(np.clip(np.asarray(probabilities, np.float64), eps, None)) / float(T)
    logp -= logp.max(axis=1, keepdims=True)
    e = np.exp(logp)
    return (e / e.sum(axis=1, keepdims=True)).astype(np.float32)


def full_metrics(y_true, probabilities, n_classes=NUM_CLASSES):
    """Every discrimination metric the manuscript reports, plus MCC."""
    p = normp(probabilities) if probabilities.shape[1] == NUM_CLASSES else np.asarray(probabilities)
    pred = np.argmax(p, axis=1)
    labels = np.arange(n_classes)

    try:
        y_bin = label_binarize(y_true, classes=labels)
        present = y_bin.sum(axis=0) > 0
        auc = float(roc_auc_score(y_bin[:, present], p[:, present],
                                  average="macro", multi_class="ovr"))
    except Exception:
        auc = float("nan")

    cm = confusion_matrix(y_true, pred, labels=labels)
    recalls, specs = [], []
    for k in labels:
        tp = cm[k, k]
        fn = cm[k, :].sum() - tp
        fp = cm[:, k].sum() - tp
        tn = cm.sum() - tp - fn - fp
        if (tp + fn) > 0:
            recalls.append(tp / (tp + fn))
        specs.append(tn / max(tn + fp, 1))

    return {
        "accuracy":          float(accuracy_score(y_true, pred)),
        "bacc":              float(np.mean(recalls)),
        "macro_precision":   float(precision_score(y_true, pred, average="macro", zero_division=0)),
        "macro_recall":      float(recall_score(y_true, pred, average="macro", zero_division=0)),
        "macro_f1":          float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "weighted_f1":       float(f1_score(y_true, pred, average="weighted", zero_division=0)),
        "macro_specificity": float(np.mean(specs)),
        "auc_macro_ovr":     auc,
        "mcc":               float(matthews_corrcoef(y_true, pred)),
    }


def metric_triplet(y_true, probabilities):
    pred = np.argmax(probabilities, axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "bacc":     float(balanced_accuracy_score(y_true, pred)),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
    }


def per_class_recall(y_true, probabilities, n_classes=NUM_CLASSES):
    pred = np.argmax(probabilities, axis=1)
    cm = confusion_matrix(y_true, pred, labels=np.arange(n_classes))
    out = {}
    for k in range(n_classes):
        tp = cm[k, k]
        fn = cm[k, :].sum() - tp
        out[k] = float(tp / (tp + fn)) if (tp + fn) > 0 else float("nan")
    return out


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=np.float64)
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=np.float64)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p_values[idx])
        adjusted[idx] = min(running, 1.0)
    return adjusted


def mcnemar_exact(y_true, prob_a, prob_b):
    a = (np.argmax(prob_a, axis=1) == y_true)
    b = (np.argmax(prob_b, axis=1) == y_true)
    a_only = int(np.sum(a & ~b))
    b_only = int(np.sum(~a & b))
    disc = a_only + b_only
    if disc == 0:
        p = 1.0
    else:
        p = float(binomtest(k=min(a_only, b_only), n=disc, p=0.5,
                            alternative="two-sided").pvalue)
    return {
        "both_correct":  int(np.sum(a & b)),
        "a_only_correct": a_only,
        "b_only_correct": b_only,
        "both_wrong":    int(np.sum(~a & ~b)),
        "discordant":    disc,
        "mcnemar_exact_p": p,
    }


def fmt_pct(x, nd=2):
    return "--" if (x is None or (isinstance(x, float) and math.isnan(x))) else f"{100*x:.{nd}f}"


def fmt_ci(lo, hi, nd=2):
    return f"({100*lo:.{nd}f}--{100*hi:.{nd}f})"


def fmt_signed(x, nd=2):
    return f"{'+' if x >= 0 else '$-$'}{abs(100*x):.{nd}f}"


def fmt_p(p):
    if p is None or (isinstance(p, float) and math.isnan(p)):
        return "--"
    return "$<$0.001" if p < 0.001 else f"{p:.3f}".rstrip("0").rstrip(".")


def write_latex(name, body, caption_hint=""):
    path = LATEX_DIR / name
    header = f"% {caption_hint}\n" if caption_hint else ""
    path.write_text(header + body.rstrip() + "\n", encoding="utf-8")
    print("  wrote", path.name)
    return path


print("Helpers ready.")

Helpers ready.


In [3]:
# ============================================================
# CELL 3 — ARTIFACT DISCOVERY (soft-fail, prints availability)
# ============================================================
WANTED = OrderedDict([
    ("final_npz",        ("final_internal_probabilities.npz",            ("reuse_only_all_model_ensemble",))),
    ("final_config",     ("final_ensemble_config.json",                  ("reuse_only_all_model_ensemble",))),
    ("stacker",          ("stacking_meta_model.joblib",                  ("reuse_only_all_model_ensemble",))),
    ("locked_ids",       ("original_80_10_10_locked_ids.csv",            ("j1",))),
    ("split_json",       ("rotation_fixed_train_val_test_split.json",    ("rotation_ssl_paperlike",))),
    ("simclr_notta",     ("multiseed_probabilities_no_tta.npz",          ("simclr_finetune_multiseed",))),
    ("dino_npz",         ("dinov2_original_split_multiseed_probabilities.npz", ("j2",))),
    ("supervised_npz",   ("supervised_resnet50_5seed_probabilities.npz", ("mbec_plain_supervised",))),
    ("val_features",     ("val_features.npz",                            ("j5",))),
    ("dino_val_feat",    ("dinov2_representative_val_features.npz",      ("j2", "j5"))),
    ("external_results", ("all_external_eight_class_results.csv",        ("reuse_only",))),
    ("pad_results",      ("PAD_ALL_ADAPTED_MODELS_AND_ENSEMBLES_COMPARISON.csv", ("pad",))),
    ("pad_da_results",   ("PAD_DOMAIN_ADAPTED_BOTH_STACKING_RESULTS.csv", ("pad",))),
    ("j5_cka",           ("pairwise_linear_cka.csv",                     ("j5",))),
    ("j5_cka_perm",      ("cka_permutation_significance.csv",            ("j5",))),
    ("j5_rescue",        ("final_model_error_rescue_summary.csv",        ("j5",))),
    ("j7_importance",    ("global_meta_source_importance.csv",           ("j7",))),
    ("j7_loso",          ("leave_one_source_out_ensemble_importance.csv", ("j7",))),
])

ART = OrderedDict()
rows = []
for key, (fname, prefer) in WANTED.items():
    hit = find_one(fname, prefer)
    ART[key] = hit
    rows.append({"key": key, "file": fname, "found": hit is not None,
                 "path": str(hit) if hit else ""})

AVAIL_DF = pd.DataFrame(rows)
display(AVAIL_DF[["key", "file", "found"]])
AVAIL_DF.to_csv(AUDIT_DIR / "artifact_availability.csv", index=False)

CRITICAL = ["final_npz", "final_config", "simclr_notta"]
missing_critical = [k for k in CRITICAL if ART[k] is None]
if missing_critical:
    raise FileNotFoundError(
        "Critical artifacts missing: " + ", ".join(missing_critical) +
        "\nAttach `confidence-gated-ensemble` and `last-part-of-the-previous-one` outputs."
    )

print(f"\nFound {int(AVAIL_DF['found'].sum())}/{len(AVAIL_DF)} artifacts.")

                 key                                               file  found
0          final_npz                   final_internal_probabilities.npz   True
1       final_config                         final_ensemble_config.json   True
2            stacker                         stacking_meta_model.joblib   True
3         locked_ids                   original_80_10_10_locked_ids.csv   True
4         split_json           rotation_fixed_train_val_test_split.json   True
5       simclr_notta                 multiseed_probabilities_no_tta.npz   True
6           dino_npz  dinov2_original_split_multiseed_probabilities.npz   True
7     supervised_npz        supervised_resnet50_5seed_probabilities.npz   True
8       val_features                                   val_features.npz   True
9      dino_val_feat             dinov2_representative_val_features.npz   True
10  external_results               all_external_eight_class_results.csv   True
11       pad_results  PAD_ALL_ADAPTED_MODELS_AND_ENS


Found 17/18 artifacts.


In [4]:
# ============================================================
# CELL 4 — LOAD LOCKED PROTOCOL, LABELS, FROZEN CONFIG
# ============================================================
with np.load(ART["final_npz"]) as z:
    Y_VAL  = np.asarray(z["y_val"],  dtype=np.int32)
    Y_TEST = np.asarray(z["y_test"], dtype=np.int32)
    FINAL_VAL   = normp(z["final_val_prob"])
    FINAL_TEST  = normp(z["final_test_prob"])
    WEIGHT_VAL  = normp(z["weighted_val_prob"])
    WEIGHT_TEST = normp(z["weighted_test_prob"])
    STACK_VAL   = normp(z["stacking_val_prob"])
    STACK_TEST  = normp(z["stacking_test_prob"])
    BIAS_VAL    = normp(z["stacking_bias_val_prob"])
    BIAS_TEST   = normp(z["stacking_bias_test_prob"])

assert len(Y_VAL)  == EXPECTED_VAL_N,  f"val n={len(Y_VAL)}"
assert len(Y_TEST) == EXPECTED_TEST_N, f"test n={len(Y_TEST)}"
assert np.allclose(FINAL_TEST, BIAS_TEST, atol=1e-6), "final != class-bias variant"

with open(ART["final_config"]) as f:
    FINAL_CONFIG = json.load(f)

MODEL_NAMES = list(FINAL_CONFIG["models"])
TEMPERATURES = OrderedDict((n, float(FINAL_CONFIG["temperatures"][n])) for n in MODEL_NAMES)
WEIGHTS      = np.array([float(FINAL_CONFIG["optimized_weights"][n]) for n in MODEL_NAMES], np.float64)
BEST_C       = float(FINAL_CONFIG["stacking_C"])
BEST_ALPHA   = float(FINAL_CONFIG["stacking_alpha"])
CLASS_BIAS   = np.array(FINAL_CONFIG["class_bias"], np.float64)
ANCHOR_NAME  = str(FINAL_CONFIG["anchor"])

TEST_COUNTS = {CLASS_NAMES[k]: int(np.sum(Y_TEST == k)) for k in range(NUM_CLASSES)}

print("Sources     :", MODEL_NAMES)
print("Temperatures:", dict(TEMPERATURES))
print("Anchor      :", ANCHOR_NAME, "| C =", BEST_C, "| alpha =", BEST_ALPHA)
print("Class bias  :", CLASS_BIAS.tolist())
print("Test counts :", TEST_COUNTS)

record("protocol.test_n", int(len(Y_TEST)))
record("protocol.val_n",  int(len(Y_VAL)))
record("protocol.class_counts_test", TEST_COUNTS)

Sources     : ['SimCLR_NoTTA', 'SimCLR_SelectedTTA', 'Rotation', 'FusionConcat', 'FusionGatedV2', 'FusionGatedV3', 'ResidualFusion']
Temperatures: {'SimCLR_NoTTA': 1.3, 'SimCLR_SelectedTTA': 1.3, 'Rotation': 1.75, 'FusionConcat': 1.75, 'FusionGatedV2': 1.75, 'FusionGatedV3': 2.0, 'ResidualFusion': 2.0}
Anchor      : SimCLR_NoTTA | C = 0.01 | alpha = 0.6
Class bias  : [-0.05000000074505806, 0.05000000074505806, -0.05000000074505806, 0.15000000596046448, 0.0, 0.0, 0.0, -0.05000000074505806]
Test counts : {'MEL': 452, 'NV': 1288, 'BCC': 333, 'AK': 86, 'BKL': 263, 'DF': 24, 'VASC': 25, 'SCC': 63}


{'MEL': 452,
 'NV': 1288,
 'BCC': 333,
 'AK': 86,
 'BKL': 263,
 'DF': 24,
 'VASC': 25,
 'SCC': 63}

In [5]:
# ============================================================
# CELL 5 — LOAD THE SEVEN FROZEN SOURCES + COMPARATORS
# ============================================================
PIPELINE_ROOT = Path(ART["simclr_notta"]).parent.parent
print("PIPELINE_ROOT:", PIPELINE_ROOT)

SOURCE_SPEC = OrderedDict([
    ("SimCLR_NoTTA",      dict(kind="npz", path=PIPELINE_ROOT / "simclr_finetune_multiseed_bacc_v2"
                               / "multiseed_probabilities_no_tta.npz")),
    ("SimCLR_SelectedTTA", dict(kind="npz", path=PIPELINE_ROOT / "simclr_finetune_multiseed_bacc_v2"
                               / "tta_selection" / "selected_multiseed_tta_probabilities.npz")),
    ("Rotation",          dict(kind="csv",
                               val=PIPELINE_ROOT / "extended_imagenet_rotation_finetune" / "imagenet_rotation_val_predictions.csv",
                               test=PIPELINE_ROOT / "extended_imagenet_rotation_finetune" / "imagenet_rotation_test_predictions.csv")),
    ("FusionConcat",      dict(kind="csv",
                               val=PIPELINE_ROOT / "fusion_concat_multiseed_rep_v2" / "full_evaluation" / "val" / "val_predictions.csv",
                               test=PIPELINE_ROOT / "fusion_concat_multiseed_rep_v2" / "full_evaluation" / "test" / "test_predictions.csv")),
    ("FusionGatedV2",     dict(kind="csv",
                               val=PIPELINE_ROOT / "fusion_gated_v2_multiseed_rep_v2" / "full_evaluation" / "val" / "val_predictions.csv",
                               test=PIPELINE_ROOT / "fusion_gated_v2_multiseed_rep_v2" / "full_evaluation" / "test" / "test_predictions.csv")),
    ("FusionGatedV3",     dict(kind="csv",
                               val=PIPELINE_ROOT / "fusion_gated_v3_multiseed_rep_v2" / "full_evaluation" / "val" / "val_predictions.csv",
                               test=PIPELINE_ROOT / "fusion_gated_v3_multiseed_rep_v2" / "full_evaluation" / "test" / "test_predictions.csv")),
    ("ResidualFusion",    dict(kind="npz", path=PIPELINE_ROOT / "residual_simclr_rotation_fusion_v1"
                               / "multiseed_probabilities.npz")),
])

VAL_PROB, TEST_PROB, load_rows = OrderedDict(), OrderedDict(), []
for name, spec in SOURCE_SPEC.items():
    try:
        if spec["kind"] == "npz":
            path = spec["path"]
            if not path.exists():
                path = find_one(path.name, (str(path.parent.name),))
            with np.load(path) as z:
                yv, yt = np.asarray(z["y_val"], np.int32), np.asarray(z["y_test"], np.int32)
                pv, pt = normp(z["val_prob"]), normp(z["test_prob"])
            src = str(path)
        else:
            vp, tp = spec["val"], spec["test"]
            if not vp.exists():
                vp = find_one("val_predictions.csv",  (spec["val"].parent.parent.parent.name,))
                tp = find_one("test_predictions.csv", (spec["test"].parent.parent.parent.name,))
            yv, pv = load_prediction_csv(vp)
            yt, pt = load_prediction_csv(tp)
            src = str(tp)

        assert np.array_equal(yv, Y_VAL),  f"{name}: val labels misaligned"
        assert np.array_equal(yt, Y_TEST), f"{name}: test labels misaligned"
        VAL_PROB[name], TEST_PROB[name] = pv, pt
        load_rows.append({"source": name, "loaded": True, "path": src})
    except Exception as exc:
        load_rows.append({"source": name, "loaded": False, "path": f"ERROR: {exc}"})

LOAD_DF = pd.DataFrame(load_rows)
display(LOAD_DF[["source", "loaded"]])
LOAD_DF.to_csv(AUDIT_DIR / "source_load_audit.csv", index=False)
if not LOAD_DF["loaded"].all():
    raise RuntimeError("Not all seven frozen sources loaded; see audit above.")

# ---- comparators ----------------------------------------------------------
DINO_VAL = DINO_TEST = None
if ART["dino_npz"]:
    with np.load(ART["dino_npz"]) as z:
        if np.array_equal(np.asarray(z["y_test"], np.int32), Y_TEST):
            DINO_TEST = normp(z["test_prob"])
            DINO_VAL  = normp(z["val_prob"]) if "val_prob" in z.files else None

SUP_VAL = SUP_TEST = None
if ART["supervised_npz"]:
    with np.load(ART["supervised_npz"]) as z:
        keys = set(z.files)
        tkey = "test_prob" if "test_prob" in keys else ("mean_test_prob" if "mean_test_prob" in keys else None)
        vkey = "val_prob"  if "val_prob"  in keys else ("mean_val_prob"  if "mean_val_prob"  in keys else None)
        if tkey and np.array_equal(np.asarray(z["y_test"], np.int32), Y_TEST):
            SUP_TEST = normp(z[tkey])
            SUP_VAL  = normp(z[vkey]) if vkey else None
        print("supervised npz keys:", sorted(keys))

if SUP_TEST is None:
    print("!! Supervised 5-seed probabilities NOT loaded — Table 5 supervised row will stay blank.")

# ---- canonical model registry used by every downstream section ------------
MODELS_TEST = OrderedDict()
MODELS_TEST["Supervised ResNet50"]        = SUP_TEST
MODELS_TEST["SimCLR No-TTA"]              = TEST_PROB["SimCLR_NoTTA"]
MODELS_TEST["SimCLR selected-TTA"]        = TEST_PROB["SimCLR_SelectedTTA"]
MODELS_TEST["DINOv2 ViT-S/14"]            = DINO_TEST
MODELS_TEST["Rotation"]                   = TEST_PROB["Rotation"]
MODELS_TEST["FusionConcat"]               = TEST_PROB["FusionConcat"]
MODELS_TEST["FusionGated V2"]             = TEST_PROB["FusionGatedV2"]
MODELS_TEST["FusionGated V3"]             = TEST_PROB["FusionGatedV3"]
MODELS_TEST["Residual Fusion"]            = TEST_PROB["ResidualFusion"]
MODELS_TEST["Weighted-average ensemble"]  = WEIGHT_TEST
MODELS_TEST["Anchor + stacking (no bias)"] = STACK_TEST
MODELS_TEST["CalStack-SSL"]               = FINAL_TEST
MODELS_TEST = OrderedDict((k, v) for k, v in MODELS_TEST.items() if v is not None)

print("\nModels available for reporting:", len(MODELS_TEST))
for k in MODELS_TEST:
    print("  -", k)

PIPELINE_ROOT: /kaggle/input/notebooks/anirbanroysudipto/last-part-of-the-previous-one/rotation_ssl_paperlike


               source  loaded
0        SimCLR_NoTTA    True
1  SimCLR_SelectedTTA    True
2            Rotation    True
3        FusionConcat    True
4       FusionGatedV2    True
5       FusionGatedV3    True
6      ResidualFusion    True

supervised npz keys: ['seeds', 'test_prob', 'val_prob', 'y_test', 'y_val']

Models available for reporting: 12
  - Supervised ResNet50
  - SimCLR No-TTA
  - SimCLR selected-TTA
  - DINOv2 ViT-S/14
  - Rotation
  - FusionConcat
  - FusionGated V2
  - FusionGated V3
  - Residual Fusion
  - Weighted-average ensemble
  - Anchor + stacking (no bias)
  - CalStack-SSL


## 1 — Table 4: locked-test performance for every model

In [6]:
# ============================================================
# CELL 6 — TABLE 4: FULL LOCKED-TEST METRICS
# ============================================================
rows = []
for name, prob in MODELS_TEST.items():
    m = full_metrics(Y_TEST, prob)
    rows.append({"Model": name, **m})

TABLE4_RAW = pd.DataFrame(rows)
TABLE4_PCT = TABLE4_RAW.copy()
for c in TABLE4_PCT.columns:
    if c != "Model":
        TABLE4_PCT[c] = 100 * TABLE4_PCT[c]

display(TABLE4_PCT.round(3))
TABLE4_RAW.to_csv(TABLE_DIR / "Table4_locked_test_metrics.csv", index=False)
TABLE4_PCT.to_csv(TABLE_DIR / "Table4_locked_test_metrics_percent.csv", index=False)

def m_of(model, metric):
    return float(TABLE4_RAW.loc[TABLE4_RAW["Model"] == model, metric].iloc[0])

record("table4.calstack_bacc",     m_of("CalStack-SSL", "bacc"))
record("table4.calstack_accuracy", m_of("CalStack-SSL", "accuracy"))
record("table4.calstack_macro_f1", m_of("CalStack-SSL", "macro_f1"))
record("table4.calstack_mcc",      m_of("CalStack-SSL", "mcc"))
if "Supervised ResNet50" in MODELS_TEST:
    record("abstract.delta_bacc_vs_supervised",
           m_of("CalStack-SSL", "bacc") - m_of("Supervised ResNet50", "bacc"),
           "percentage points when x100")

                          Model  accuracy    bacc  macro_precision  macro_recall  macro_f1  weighted_f1  macro_specificity  auc_macro_ovr     mcc
0           Supervised ResNet50    84.846  80.170           83.578        80.170    81.477       84.685             97.204         97.595  77.530
1                 SimCLR No-TTA    84.451  81.073           82.167        81.073    81.443       84.532             97.261         97.788  77.228
2           SimCLR selected-TTA    84.570  80.587           81.262        80.587    80.687       84.640             97.299         97.799  77.421
3               DINOv2 ViT-S/14    79.637  80.430           77.830        80.430    78.834       80.108             96.628         97.401  71.141
4                      Rotation    79.519  73.744           74.141        73.744    73.731       79.481             96.306         96.582  69.757
5                  FusionConcat    82.281  78.726           72.216        78.726    75.050       82.405             96.962  

## 2 — Bootstrap CIs (5,000 replicates) for every model and every pairwise difference

In [7]:
# ============================================================
# CELL 7 — CLASS-STRATIFIED PAIRED BOOTSTRAP (all models at once)
# ============================================================
rng = np.random.default_rng(BOOTSTRAP_SEED)
class_idx = {c: np.flatnonzero(Y_TEST == c) for c in range(NUM_CLASSES)}
for c in range(NUM_CLASSES):
    assert len(class_idx[c]) > 0, f"class {c} absent from test set"

def draw_indices():
    return np.concatenate([
        rng.choice(class_idx[c], size=len(class_idx[c]), replace=True)
        for c in range(NUM_CLASSES)
    ])

METRIC_KEYS = ["accuracy", "bacc", "macro_f1"]
BOOT = {name: {k: np.empty(N_BOOTSTRAP, np.float32) for k in METRIC_KEYS}
        for name in MODELS_TEST}

t0 = time.time()
for b in range(N_BOOTSTRAP):
    idx = draw_indices()
    yb = Y_TEST[idx]
    for name, prob in MODELS_TEST.items():
        mb = metric_triplet(yb, prob[idx])
        for k in METRIC_KEYS:
            BOOT[name][k][b] = mb[k]
    if (b + 1) % 1000 == 0:
        print(f"  bootstrap {b+1}/{N_BOOTSTRAP}  ({time.time()-t0:.0f}s)")

print(f"Bootstrap complete in {time.time()-t0:.0f}s")

# ---- absolute CI table ----------------------------------------------------
ci_rows = []
for name, prob in MODELS_TEST.items():
    obs = metric_triplet(Y_TEST, prob)
    for k in METRIC_KEYS:
        v = BOOT[name][k]
        lo, hi = np.percentile(v, [2.5, 97.5])
        ci_rows.append({
            "model": name, "metric": k, "observed": obs[k],
            "bootstrap_mean": float(v.mean()),
            "ci95_low": float(lo), "ci95_high": float(hi),
            "n_bootstrap": N_BOOTSTRAP,
        })

CI_DF = pd.DataFrame(ci_rows)
CI_DF.to_csv(TABLE_DIR / "bootstrap_95ci_all_models.csv", index=False)

CI_PCT = CI_DF.copy()
for c in ["observed", "bootstrap_mean", "ci95_low", "ci95_high"]:
    CI_PCT[c] = 100 * CI_PCT[c]
CI_PCT.to_csv(TABLE_DIR / "bootstrap_95ci_all_models_percent.csv", index=False)
display(CI_PCT.loc[CI_PCT.metric == "bacc"].round(3))

def ci_of(model, metric="bacc"):
    r = CI_DF[(CI_DF.model == model) & (CI_DF.metric == metric)].iloc[0]
    return float(r.ci95_low), float(r.ci95_high)

  bootstrap 1000/5000  (35s)
  bootstrap 2000/5000  (71s)
  bootstrap 3000/5000  (106s)
  bootstrap 4000/5000  (142s)
  bootstrap 5000/5000  (177s)
Bootstrap complete in 177s


                          model metric  observed  bootstrap_mean  ci95_low  ci95_high  n_bootstrap
1           Supervised ResNet50   bacc    80.170          80.174    77.081     83.069         5000
4                 SimCLR No-TTA   bacc    81.073          81.060    77.984     83.931         5000
7           SimCLR selected-TTA   bacc    80.587          80.580    77.436     83.491         5000
10              DINOv2 ViT-S/14   bacc    80.430          80.407    77.514     83.177         5000
13                     Rotation   bacc    73.744          73.726    70.252     77.075         5000
16                 FusionConcat   bacc    78.726          78.718    75.549     81.864         5000
19               FusionGated V2   bacc    77.223          77.200    73.846     80.546         5000
22               FusionGated V3   bacc    77.425          77.417    74.138     80.584         5000
25              Residual Fusion   bacc    78.924          78.936    75.782     82.053         5000
28    Weig

In [8]:
# ============================================================
# CELL 8 — PAIRED DIFFERENCES + EXACT McNEMAR (Table 5)
# ============================================================
REFERENCE = "CalStack-SSL"
COMPARATORS = [m for m in MODELS_TEST if m in [
    "SimCLR No-TTA", "DINOv2 ViT-S/14", "Rotation",
    "Weighted-average ensemble", "Supervised ResNet50",
]]

paired_rows = []
ref_obs = metric_triplet(Y_TEST, MODELS_TEST[REFERENCE])
for comp in COMPARATORS:
    comp_obs = metric_triplet(Y_TEST, MODELS_TEST[comp])
    for k in METRIC_KEYS:
        diff = BOOT[REFERENCE][k] - BOOT[comp][k]
        lo, hi = np.percentile(diff, [2.5, 97.5])
        n_nonpos = int(np.sum(diff <= 0)); n_nonneg = int(np.sum(diff >= 0))
        p_two = min(1.0, 2.0 * min((n_nonpos + 1) / (N_BOOTSTRAP + 1),
                                   (n_nonneg + 1) / (N_BOOTSTRAP + 1)))
        paired_rows.append({
            "comparator": comp, "metric": k,
            "observed_difference": ref_obs[k] - comp_obs[k],
            "bootstrap_mean_difference": float(diff.mean()),
            "ci95_low": float(lo), "ci95_high": float(hi),
            "prob_reference_greater": float(np.mean(diff > 0)),
            "two_sided_bootstrap_p": float(p_two),
        })

PAIRED_DF = pd.DataFrame(paired_rows)
PAIRED_DF.to_csv(TABLE_DIR / "paired_bootstrap_calstack_vs_comparators.csv", index=False)

PAIRED_PCT = PAIRED_DF.copy()
for c in ["observed_difference", "bootstrap_mean_difference", "ci95_low", "ci95_high"]:
    PAIRED_PCT[c] = 100 * PAIRED_PCT[c]
PAIRED_PCT.to_csv(TABLE_DIR / "paired_bootstrap_percent.csv", index=False)
display(PAIRED_PCT.loc[PAIRED_PCT.metric == "bacc"].round(3))

# ---- McNemar --------------------------------------------------------------
mc_rows = []
for comp in COMPARATORS:
    d = mcnemar_exact(Y_TEST, MODELS_TEST[REFERENCE], MODELS_TEST[comp])
    mc_rows.append({"comparator": comp, **d})
MCNEMAR_DF = pd.DataFrame(mc_rows)
MCNEMAR_DF["holm_adjusted_p"] = holm_adjust(MCNEMAR_DF["mcnemar_exact_p"].to_numpy())
MCNEMAR_DF["significant_holm_0.05"] = MCNEMAR_DF["holm_adjusted_p"] < 0.05
MCNEMAR_DF.to_csv(TABLE_DIR / "mcnemar_calstack_vs_comparators.csv", index=False)
display(MCNEMAR_DF)

def paired_of(comp, metric="bacc"):
    r = PAIRED_DF[(PAIRED_DF.comparator == comp) & (PAIRED_DF.metric == metric)]
    return None if len(r) == 0 else r.iloc[0]

def mcnemar_of(comp):
    r = MCNEMAR_DF[MCNEMAR_DF.comparator == comp]
    return None if len(r) == 0 else r.iloc[0]

for comp in COMPARATORS:
    r = paired_of(comp, "bacc")
    record(f"table5.{comp}.dbacc", float(r.observed_difference))
    record(f"table5.{comp}.dbacc_ci", [float(r.ci95_low), float(r.ci95_high)])
    ra = paired_of(comp, "accuracy")
    record(f"table5.{comp}.dacc", float(ra.observed_difference))
    record(f"table5.{comp}.dacc_ci", [float(ra.ci95_low), float(ra.ci95_high)])
    m = mcnemar_of(comp)
    record(f"table5.{comp}.mcnemar_holm_p", float(m.holm_adjusted_p))

                   comparator metric  observed_difference  bootstrap_mean_difference  ci95_low  ci95_high  prob_reference_greater  two_sided_bootstrap_p
1         Supervised ResNet50   bacc                3.785                      3.781     1.142      6.520                   0.997                  0.007
4               SimCLR No-TTA   bacc                2.882                      2.894     0.795      5.221                   0.999                  0.003
7             DINOv2 ViT-S/14   bacc                3.525                      3.548     0.849      6.240                   0.995                  0.010
10                   Rotation   bacc               10.211                     10.228     7.002     13.591                   1.000                  0.000
13  Weighted-average ensemble   bacc                2.926                      2.936     1.002      5.189                   0.999                  0.002

                  comparator  both_correct  a_only_correct  b_only_correct  both_wrong  discordant  mcnemar_exact_p  holm_adjusted_p  significant_holm_0.05
0        Supervised ResNet50          2011             115             139         269         254     1.488346e-01     2.976691e-01                  False
1              SimCLR No-TTA          2072              54              68         340         122     2.390888e-01     2.976691e-01                  False
2            DINOv2 ViT-S/14          1866             260             152         256         412     1.152990e-07     4.611959e-07                   True
3                   Rotation          1898             228             117         291         345     2.331752e-09     1.165876e-08                   True
4  Weighted-average ensemble          2083              43              73         335         116     6.831631e-03     2.049489e-02                   True

## 3 — Table 6: per-class recall + MCC

In [9]:
# ============================================================
# CELL 9 — TABLE 6: PER-CLASS RECALL + MCC
# ============================================================
PER_CLASS_MODELS = [m for m in
                    ["Supervised ResNet50", "SimCLR No-TTA", "Rotation", "CalStack-SSL"]
                    if m in MODELS_TEST]

pc_rows = []
for cls_id, cls in enumerate(CLASS_NAMES):
    row = {"Class": cls, "n": int(np.sum(Y_TEST == cls_id))}
    for m in PER_CLASS_MODELS:
        row[m] = per_class_recall(Y_TEST, MODELS_TEST[m])[cls_id]
    pc_rows.append(row)

macro_row = {"Class": "Macro (BACC)", "n": int(len(Y_TEST))}
mcc_row   = {"Class": "MCC",          "n": int(len(Y_TEST))}
for m in PER_CLASS_MODELS:
    fm = full_metrics(Y_TEST, MODELS_TEST[m])
    macro_row[m] = fm["bacc"]
    mcc_row[m]   = fm["mcc"]
pc_rows += [macro_row, mcc_row]

TABLE6 = pd.DataFrame(pc_rows)
TABLE6_PCT = TABLE6.copy()
for m in PER_CLASS_MODELS:
    TABLE6_PCT[m] = 100 * TABLE6_PCT[m]
display(TABLE6_PCT.round(2))
TABLE6.to_csv(TABLE_DIR / "Table6_per_class_recall.csv", index=False)
TABLE6_PCT.to_csv(TABLE_DIR / "Table6_per_class_recall_percent.csv", index=False)

# Deltas that the Discussion quotes verbatim.
if "SimCLR No-TTA" in PER_CLASS_MODELS:
    base = "SimCLR No-TTA"
    deltas = {}
    for cls_id, cls in enumerate(CLASS_NAMES):
        d = (per_class_recall(Y_TEST, MODELS_TEST["CalStack-SSL"])[cls_id]
             - per_class_recall(Y_TEST, MODELS_TEST[base])[cls_id])
        deltas[cls] = float(d)
    DELTA_DF = pd.DataFrame([{"class": k, "delta_recall_pp": 100 * v} for k, v in deltas.items()])
    display(DELTA_DF.round(2))
    DELTA_DF.to_csv(TABLE_DIR / "Table6B_per_class_recall_delta_vs_simclr.csv", index=False)
    for cls, v in deltas.items():
        record(f"classwise.delta_recall_vs_simclr.{cls}", v)
    record("classwise.max_abs_delta_MEL_NV",
           float(max(abs(deltas["MEL"]), abs(deltas["NV"]))))

          Class     n  Supervised ResNet50  SimCLR No-TTA  Rotation  CalStack-SSL
0           MEL   452                72.35          76.11     66.59         74.78
1            NV  1288                92.39          89.60     88.20         88.04
2           BCC   333                89.49          86.79     81.38         86.79
3            AK    86                69.77          76.74     61.63         79.07
4           BKL   263                70.34          74.90     65.40         74.90
5            DF    24                70.83          66.67     62.50         79.17
6          VASC    25               100.00         100.00     96.00        100.00
7           SCC    63                76.19          77.78     68.25         88.89
8  Macro (BACC)  2534                80.17          81.07     73.74         83.95
9           MCC  2534                77.53          77.23     69.76         76.74

  class  delta_recall_pp
0   MEL            -1.33
1    NV            -1.55
2   BCC             0.00
3    AK             2.33
4   BKL             0.00
5    DF            12.50
6  VASC             0.00
7   SCC            11.11

## 4 — Calibration, reliability, selective prediction

In [10]:
# ============================================================
# CELL 10 — CALIBRATION METRICS + VALIDATION TEMPERATURE
# ============================================================
from scipy.optimize import minimize_scalar

def nll(y, p, eps=1e-12):
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], eps, None))))

def brier(y, p):
    onehot = np.zeros_like(p); onehot[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((p - onehot) ** 2, axis=1)))

def ece_metrics(y, p, n_bins=ECE_BINS):
    conf = p.max(axis=1); pred = p.argmax(axis=1); correct = (pred == y).astype(np.float64)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = mce = 0.0; bin_rows = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if m.sum() == 0:
            bin_rows.append({"bin_low": lo, "bin_high": hi, "n": 0,
                             "accuracy": np.nan, "confidence": np.nan}); continue
        acc, cf = correct[m].mean(), conf[m].mean()
        gap = abs(acc - cf)
        ece += (m.sum() / len(y)) * gap
        mce = max(mce, gap)
        bin_rows.append({"bin_low": lo, "bin_high": hi, "n": int(m.sum()),
                         "accuracy": float(acc), "confidence": float(cf)})
    # adaptive (equal-mass) ECE
    order = np.argsort(conf); aece = 0.0
    for chunk in np.array_split(order, n_bins):
        if len(chunk) == 0: continue
        aece += (len(chunk) / len(y)) * abs(correct[chunk].mean() - conf[chunk].mean())
    return {"ece": float(ece), "adaptive_ece": float(aece), "mce": float(mce),
            "mean_confidence": float(conf.mean()), "accuracy": float(correct.mean())}, \
           pd.DataFrame(bin_rows)

def calibration_row(tag, y, p):
    e, _ = ece_metrics(y, p)
    return {"model": tag, **e, "nll": nll(y, p), "brier": brier(y, p)}

# Temperature selected on VALIDATION only.
def val_nll(T):
    return nll(Y_VAL, temperature_scale(FINAL_VAL, T))

T_LO, T_HI = 0.20, 6.0
opt = minimize_scalar(val_nll, bounds=(T_LO, T_HI), method="bounded")
T_STAR = float(opt.x)
print(f"Validation-selected temperature T* = {T_STAR:.4f}")
if abs(T_STAR - T_LO) < 1e-3 or abs(T_STAR - T_HI) < 1e-3:
    print("!! WARNING: T* hit a search bound; widen (T_LO, T_HI) and re-run.")

FINAL_TEST_TS = temperature_scale(FINAL_TEST, T_STAR)
label_change = int(np.sum(FINAL_TEST.argmax(1) != FINAL_TEST_TS.argmax(1)))
print("Predicted labels changed by temperature scaling:", label_change)

cal_rows = [calibration_row("CalStack-SSL (raw)", Y_TEST, FINAL_TEST),
            calibration_row("CalStack-SSL (+ val temperature)", Y_TEST, FINAL_TEST_TS)]
for m in ["SimCLR No-TTA", "Supervised ResNet50", "DINOv2 ViT-S/14", "Rotation"]:
    if m in MODELS_TEST:
        cal_rows.append(calibration_row(m, Y_TEST, MODELS_TEST[m]))

CALIB_DF = pd.DataFrame(cal_rows)
CALIB_PCT = CALIB_DF.copy()
for c in ["ece", "adaptive_ece", "mce", "mean_confidence", "accuracy"]:
    CALIB_PCT[c] = 100 * CALIB_PCT[c]
display(CALIB_PCT.round(4))
CALIB_DF.to_csv(TABLE_DIR / "Table_calibration.csv", index=False)
CALIB_PCT.to_csv(TABLE_DIR / "Table_calibration_percent.csv", index=False)

raw = CALIB_DF.iloc[0]; scaled = CALIB_DF.iloc[1]
record("calibration.temperature", T_STAR)
record("calibration.ece_raw", float(raw.ece))
record("calibration.ece_scaled", float(scaled.ece))
record("calibration.mean_conf_raw", float(raw.mean_confidence))
record("calibration.mean_conf_scaled", float(scaled.mean_confidence))
record("calibration.labels_changed", label_change)

json.dump({"temperature": T_STAR, "selected_on": "validation", "labels_changed": label_change},
          open(AUDIT_DIR / "validation_temperature_selection.json", "w"), indent=2)

Validation-selected temperature T* = 0.8658
Predicted labels changed by temperature scaling: 0


                              model     ece  adaptive_ece      mce  mean_confidence  accuracy     nll   brier
0                CalStack-SSL (raw)  3.7216        3.4710  40.6083          80.6542   83.8990  0.4627  0.2318
1  CalStack-SSL (+ val temperature)  1.9271        1.5419  10.9990          83.8416   83.8990  0.4529  0.2298
2                     SimCLR No-TTA  2.6642        1.5759  26.2310          85.7030   84.4515  0.4662  0.2274
3               Supervised ResNet50  5.2029        5.0352  71.2356          89.8812   84.8461  0.5068  0.2270
4                   DINOv2 ViT-S/14  1.9809        1.5468  23.9485          80.0820   79.6369  0.5235  0.2827
5                          Rotation  9.5437        9.1189  31.8407          88.6366   79.5185  0.6524  0.3079

In [11]:
# ============================================================
# CELL 11 — RELIABILITY DIAGRAM (Fig 3)
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10, "figure.dpi": 120,
                     "savefig.dpi": 600, "savefig.bbox": "tight"})

_, bins_raw    = ece_metrics(Y_TEST, FINAL_TEST)
_, bins_scaled = ece_metrics(Y_TEST, FINAL_TEST_TS)
bins_raw.to_csv(TABLE_DIR / "reliability_bins_raw.csv", index=False)
bins_scaled.to_csv(TABLE_DIR / "reliability_bins_scaled.csv", index=False)

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(6.2, 6.4),
                               gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
ax0.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
for df, lab, mk in [(bins_raw, f"Raw (ECE {100*raw.ece:.2f}%)", "o"),
                    (bins_scaled, f"Temperature scaled (ECE {100*scaled.ece:.2f}%)", "s")]:
    d = df.dropna()
    ax0.plot(d.confidence, d.accuracy, marker=mk, lw=1.6, ms=5, label=lab)
ax0.set_ylabel("Accuracy"); ax0.set_xlim(0, 1); ax0.set_ylim(0, 1)
ax0.legend(loc="upper left", frameon=False, fontsize=9); ax0.grid(alpha=0.25)

centres = (bins_raw.bin_low + bins_raw.bin_high) / 2
ax1.bar(centres, bins_raw.n, width=1.0 / ECE_BINS * 0.9, color="0.6", edgecolor="0.3", lw=0.4)
ax1.set_xlabel("Confidence"); ax1.set_ylabel("Count"); ax1.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig3_reliability.png"); plt.close()
print("Fig3_reliability.png written")

Fig3_reliability.png written


In [12]:
# ============================================================
# CELL 12 — SELECTIVE PREDICTION / RISK-COVERAGE (Fig 4)
# ============================================================
def selective_table(y, p, coverages=COVERAGES):
    conf = p.max(axis=1); order = np.argsort(-conf); rows = []
    for cov in coverages:
        k = max(int(round(cov * len(y))), NUM_CLASSES)
        keep = order[:k]
        yk, pk = y[keep], p[keep]
        pred = pk.argmax(1)
        acc = float(np.mean(pred == yk))
        present = np.unique(yk)
        cm = confusion_matrix(yk, pred, labels=np.arange(NUM_CLASSES))
        recs = [cm[c, c] / cm[c, :].sum() for c in present if cm[c, :].sum() > 0]
        rows.append({
            "coverage": float(k / len(y)), "retained_n": int(k),
            "confidence_threshold": float(conf[order[k - 1]]),
            "accuracy": acc, "bacc": float(np.mean(recs)), "risk": 1.0 - acc,
            "classes_present": int(len(present)),
        })
    return pd.DataFrame(rows)

SEL_DF = selective_table(Y_TEST, FINAL_TEST_TS)
SEL_PCT = SEL_DF.copy()
for c in ["coverage", "accuracy", "bacc", "risk"]:
    SEL_PCT[c] = 100 * SEL_PCT[c]
display(SEL_PCT.round(3))
SEL_DF.to_csv(TABLE_DIR / "Table_selective_prediction.csv", index=False)
SEL_PCT.to_csv(TABLE_DIR / "Table_selective_prediction_percent.csv", index=False)

# AURC over the full coverage sweep
conf = FINAL_TEST_TS.max(axis=1); order = np.argsort(-conf)
correct_sorted = (FINAL_TEST_TS.argmax(1) == Y_TEST)[order]
cum_risk = 1.0 - np.cumsum(correct_sorted) / np.arange(1, len(Y_TEST) + 1)
cov_grid = np.arange(1, len(Y_TEST) + 1) / len(Y_TEST)
_trap = getattr(np, "trapezoid", None) or np.trapz
AURC = float(_trap(cum_risk, cov_grid))
print("AURC:", round(AURC, 5))
pd.DataFrame([{"aurc": AURC, "n": len(Y_TEST)}]).to_csv(TABLE_DIR / "risk_coverage_aurc.csv", index=False)

fig, (a0, a1) = plt.subplots(1, 2, figsize=(9.4, 3.9))
a0.plot(cov_grid, cum_risk, lw=1.8)
a0.set_xlabel("Coverage"); a0.set_ylabel("Selective risk (1 $-$ accuracy)")
a0.set_title(f"Risk--coverage (AURC = {AURC:.4f})", fontsize=10); a0.grid(alpha=0.25)
a0.invert_xaxis()

a1.plot(100 * SEL_DF.coverage, 100 * SEL_DF.accuracy, marker="o", ms=4, label="Accuracy")
a1.plot(100 * SEL_DF.coverage, 100 * SEL_DF.bacc, marker="s", ms=4, label="Balanced accuracy")
a1.set_xlabel("Coverage (%)"); a1.set_ylabel("Performance on retained cases (%)")
a1.legend(frameon=False, fontsize=9); a1.grid(alpha=0.25); a1.invert_xaxis()
plt.tight_layout(); plt.savefig(FIG_DIR / "Fig4_risk_coverage.png"); plt.close()
print("Fig4_risk_coverage.png written")

def sel_at(cov):
    i = int(np.argmin(np.abs(SEL_DF.coverage.to_numpy() - cov)))
    return SEL_DF.iloc[i]

s90, s50 = sel_at(0.90), sel_at(0.50)
record("selective.acc_at_90", float(s90.accuracy))
record("selective.bacc_at_90", float(s90.bacc))
record("selective.acc_at_50", float(s50.accuracy))
record("selective.bacc_at_50", float(s50.bacc))
record("selective.threshold_at_50", float(s50.confidence_threshold))
record("selective.aurc", AURC)

   coverage  retained_n  confidence_threshold  accuracy    bacc    risk  classes_present
0   100.000        2534                 0.270    83.899  83.955  16.101                8
1    94.988        2407                 0.477    86.290  86.453  13.710                8
2    90.016        2281                 0.550    88.163  88.334  11.837                8
3    79.992        2027                 0.678    92.205  91.662   7.795                8
4    70.008        1774                 0.785    93.912  93.922   6.088                8
5    59.984        1520                 0.866    95.987  95.605   4.013                8
6    50.000        1267                 0.917    97.474  96.311   2.526                8
7    40.016        1014                 0.950    98.521  96.901   1.479                8
8    29.992         760                 0.972    99.474  99.135   0.526                8

AURC: 0.04162
Fig4_risk_coverage.png written


0.04161662506515365

## 5 — Representation complementarity: CKA, disagreement, error rescue

In [13]:
# ============================================================
# CELL 13 — LINEAR CKA + PERMUTATION TEST (Fig 5a)
# ============================================================
def linear_cka(X, Y):
    X = np.asarray(X, np.float64); Y = np.asarray(Y, np.float64)
    X = X - X.mean(0, keepdims=True); Y = Y - Y.mean(0, keepdims=True)
    xty = X.T @ Y
    num = float(np.sum(xty ** 2))
    den = float(np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro"))
    return num / den if den > 0 else float("nan")

FEATS = {}
if ART["val_features"]:
    with np.load(ART["val_features"]) as z:
        for want, keys in [("SimCLR", ["simclr", "simclr_val", "simclr_features"]),
                           ("Rotation", ["rotation", "rotation_val", "rotation_features"])]:
            for k in keys:
                if k in z.files:
                    FEATS[want] = np.asarray(z[k], np.float32); break
        print("val_features.npz keys:", sorted(z.files))
if ART["dino_val_feat"]:
    with np.load(ART["dino_val_feat"]) as z:
        for k in ["features", "val_features", "dinov2_val_features", "dino_val"]:
            if k in z.files:
                FEATS["DINOv2"] = np.asarray(z[k], np.float32); break

print("Feature matrices loaded:", {k: v.shape for k, v in FEATS.items()})

CKA_DF = None
if len(FEATS) >= 2:
    N_PERM, PERM_SUBSET, PERM_SEED = 200, 800, 42
    prng = np.random.default_rng(PERM_SEED)
    pairs = [("SimCLR", "Rotation"), ("SimCLR", "DINOv2"), ("Rotation", "DINOv2")]
    rows = []
    for a, b in pairs:
        if a not in FEATS or b not in FEATS:
            continue
        Xa, Xb = FEATS[a], FEATS[b]
        n = min(len(Xa), len(Xb))
        obs = linear_cka(Xa[:n], Xb[:n])
        sub = prng.choice(n, size=min(PERM_SUBSET, n), replace=False)
        obs_sub = linear_cka(Xa[sub], Xb[sub])
        null = np.array([linear_cka(Xa[sub], Xb[prng.permutation(sub)]) for _ in range(N_PERM)])
        p_perm = float((np.sum(null >= obs_sub) + 1) / (N_PERM + 1))
        rows.append({"representation_a": a, "representation_b": b, "linear_cka": obs,
                     "observed_cka_subset": obs_sub, "null_mean": float(null.mean()),
                     "null_std": float(null.std()), "permutation_p": p_perm,
                     "n_samples": int(n), "n_permutations": N_PERM})
    CKA_DF = pd.DataFrame(rows)
elif ART["j5_cka"]:
    print("Feature files unavailable — reusing J5 CKA CSV.")
    CKA_DF = pd.read_csv(ART["j5_cka"])
    if ART["j5_cka_perm"]:
        perm = pd.read_csv(ART["j5_cka_perm"])
        keep = [c for c in ["representation_a", "representation_b", "permutation_p"] if c in perm.columns]
        CKA_DF = CKA_DF.merge(perm[keep], on=["representation_a", "representation_b"], how="left")

if CKA_DF is not None:
    display(CKA_DF.round(4))
    CKA_DF.to_csv(TABLE_DIR / "Table_pairwise_cka.csv", index=False)
    for _, r in CKA_DF.iterrows():
        record(f"cka.{r.representation_a}_{r.representation_b}", float(r.linear_cka))
        if "permutation_p" in r and not pd.isna(r.get("permutation_p")):
            record(f"cka.{r.representation_a}_{r.representation_b}.p", float(r["permutation_p"]))
else:
    print("!! No CKA computed — attach journal-extension-j5 output.")

val_features.npz keys: ['labels', 'rotation_features', 'simclr_features']
Feature matrices loaded: {'SimCLR': (2533, 2048), 'Rotation': (2533, 2048), 'DINOv2': (2533, 384)}


  representation_a representation_b  linear_cka  observed_cka_subset  null_mean  null_std  permutation_p  n_samples  n_permutations
0           SimCLR         Rotation      0.2484               0.2491     0.0088    0.0018          0.005       2533             200
1           SimCLR           DINOv2      0.6069               0.6053     0.0349    0.0017          0.005       2533             200
2         Rotation           DINOv2      0.1459               0.1345     0.0059    0.0016          0.005       2533             200

In [14]:
# ============================================================
# CELL 14 — PREDICTION DISAGREEMENT + ERROR RESCUE (Fig 5b)
# ============================================================
DIS_MODELS = [m for m in ["SimCLR No-TTA", "Rotation", "DINOv2 ViT-S/14",
                          "Supervised ResNet50", "CalStack-SSL"] if m in MODELS_TEST]
preds = {m: MODELS_TEST[m].argmax(1) for m in DIS_MODELS}
D = np.zeros((len(DIS_MODELS), len(DIS_MODELS)))
for i, a in enumerate(DIS_MODELS):
    for j, b in enumerate(DIS_MODELS):
        D[i, j] = float(np.mean(preds[a] != preds[b]))

DIS_DF = pd.DataFrame(100 * D, index=DIS_MODELS, columns=DIS_MODELS)
display(DIS_DF.round(2))
DIS_DF.to_csv(TABLE_DIR / "Table_prediction_disagreement_percent.csv")

# ---- error rescue relative to the SimCLR anchor ---------------------------
RESCUE_DF = None
if "SimCLR No-TTA" in MODELS_TEST:
    anchor_ok = (MODELS_TEST["SimCLR No-TTA"].argmax(1) == Y_TEST)
    final_ok  = (MODELS_TEST["CalStack-SSL"].argmax(1) == Y_TEST)
    rescued   = int(np.sum(~anchor_ok & final_ok))
    broken    = int(np.sum(anchor_ok & ~final_ok))
    rows = [{"source_model": "SimCLR No-TTA (anchor)",
             "source_total_errors": int(np.sum(~anchor_ok)),
             "errors_rescued_by_final": rescued,
             "rescue_rate_among_source_errors": rescued / max(int(np.sum(~anchor_ok)), 1),
             "source_correct_but_final_wrong": broken,
             "net_gain": rescued - broken}]
    for m in ["Rotation", "DINOv2 ViT-S/14", "Supervised ResNet50"]:
        if m not in MODELS_TEST: continue
        ok = (MODELS_TEST[m].argmax(1) == Y_TEST)
        r = int(np.sum(~ok & final_ok)); b = int(np.sum(ok & ~final_ok))
        rows.append({"source_model": m, "source_total_errors": int(np.sum(~ok)),
                     "errors_rescued_by_final": r,
                     "rescue_rate_among_source_errors": r / max(int(np.sum(~ok)), 1),
                     "source_correct_but_final_wrong": b, "net_gain": r - b})
    RESCUE_DF = pd.DataFrame(rows)
    display(RESCUE_DF.round(4))
    RESCUE_DF.to_csv(TABLE_DIR / "Table_error_rescue.csv", index=False)
    record("rescue.anchor_errors_rescued", rescued)
    record("rescue.anchor_correct_broken", broken)
    record("rescue.net_gain", rescued - broken)

    # class-wise rescue
    cw = []
    for c, cls in enumerate(CLASS_NAMES):
        m = (Y_TEST == c)
        cw.append({"class": cls, "n": int(m.sum()),
                   "rescued": int(np.sum(m & ~anchor_ok & final_ok)),
                   "broken": int(np.sum(m & anchor_ok & ~final_ok))})
    CW_RESCUE = pd.DataFrame(cw)
    CW_RESCUE["net"] = CW_RESCUE.rescued - CW_RESCUE.broken
    display(CW_RESCUE)
    CW_RESCUE.to_csv(TABLE_DIR / "Table_classwise_error_rescue.csv", index=False)

# ---- Fig 5: two-panel CKA + disagreement ---------------------------------
fig, (a0, a1) = plt.subplots(1, 2, figsize=(10.2, 4.2))

if CKA_DF is not None:
    reps = ["SimCLR", "Rotation", "DINOv2"]
    M = np.eye(3)
    for _, r in CKA_DF.iterrows():
        if r.representation_a in reps and r.representation_b in reps:
            i, j = reps.index(r.representation_a), reps.index(r.representation_b)
            M[i, j] = M[j, i] = float(r.linear_cka)
    im = a0.imshow(M, vmin=0, vmax=1, cmap="viridis")
    a0.set_xticks(range(3), reps); a0.set_yticks(range(3), reps)
    for i in range(3):
        for j in range(3):
            a0.text(j, i, f"{M[i,j]:.3f}", ha="center", va="center",
                    color="white" if M[i, j] < 0.6 else "black", fontsize=9)
    fig.colorbar(im, ax=a0, fraction=0.046)
    a0.set_title("a  Linear CKA (validation representations)", fontsize=10, loc="left")
else:
    a0.text(0.5, 0.5, "CKA unavailable", ha="center"); a0.axis("off")

short = [m.replace(" No-TTA", "").replace(" ViT-S/14", "").replace(" ResNet50", "")
         for m in DIS_MODELS]
im2 = a1.imshow(100 * D, cmap="magma")
a1.set_xticks(range(len(short)), short, rotation=35, ha="right")
a1.set_yticks(range(len(short)), short)
for i in range(len(short)):
    for j in range(len(short)):
        a1.text(j, i, f"{100*D[i,j]:.1f}", ha="center", va="center",
                color="white" if 100 * D[i, j] < 18 else "black", fontsize=8)
fig.colorbar(im2, ax=a1, fraction=0.046, label="Disagreement (%)")
a1.set_title("b  Pairwise prediction disagreement", fontsize=10, loc="left")
plt.tight_layout(); plt.savefig(FIG_DIR / "Fig5_complementarity.png"); plt.close()
print("Fig5_complementarity.png written")

                     SimCLR No-TTA  Rotation  DINOv2 ViT-S/14  Supervised ResNet50  CalStack-SSL
SimCLR No-TTA                 0.00     17.72            18.71                10.73          6.27
Rotation                     17.72      0.00            20.01                16.54         15.59
DINOv2 ViT-S/14              18.71     20.01             0.00                18.35         18.63
Supervised ResNet50          10.73     16.54            18.35                 0.00         11.88
CalStack-SSL                  6.27     15.59            18.63                11.88          0.00

             source_model  source_total_errors  errors_rescued_by_final  rescue_rate_among_source_errors  source_correct_but_final_wrong  net_gain
0  SimCLR No-TTA (anchor)                  394                       54                           0.1371                              68       -14
1                Rotation                  519                      228                           0.4393                             117       111
2         DINOv2 ViT-S/14                  516                      260                           0.5039                             152       108
3     Supervised ResNet50                  384                      115                           0.2995                             139       -24

  class     n  rescued  broken  net
0   MEL   452        9      15   -6
1    NV  1288       10      30  -20
2   BCC   333        9       9    0
3    AK    86        6       4    2
4   BKL   263       10      10    0
5    DF    24        3       0    3
6  VASC    25        0       0    0
7   SCC    63        7       0    7

Fig5_complementarity.png written


## 6 — Stacking source importance, LOSO, and validation-selected pruning

In [15]:
# ============================================================
# CELL 15 — REFIT STACK, SOURCE IMPORTANCE, LOSO (Fig 6)
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

CAL_VAL  = OrderedDict((n, temperature_scale(VAL_PROB[n],  TEMPERATURES[n])) for n in MODEL_NAMES)
CAL_TEST = OrderedDict((n, temperature_scale(TEST_PROB[n], TEMPERATURES[n])) for n in MODEL_NAMES)

def meta_features(cal, names, eps=1e-12):
    blocks = []
    for n in names:
        p = np.asarray(cal[n], np.float64)
        logp = np.log(np.clip(p, eps, None))
        conf = p.max(1, keepdims=True)
        ent = (-(p * logp).sum(1, keepdims=True))
        blocks += [p, logp, conf, ent]
    return np.hstack(blocks)

def fit_stack(names, C=BEST_C, seed=42):
    Xv = meta_features(CAL_VAL, names)
    clf = Pipeline([("sc", StandardScaler()),
                    ("lr", LogisticRegression(C=C, max_iter=4000,
                                              class_weight="balanced",
                                              multi_class="multinomial", random_state=seed))])
    clf.fit(Xv, Y_VAL)
    return clf

def apply_pipeline(clf, names, anchor="SimCLR_NoTTA", alpha=BEST_ALPHA, bias=CLASS_BIAS):
    out = {}
    for tag, cal, y in [("val", CAL_VAL, Y_VAL), ("test", CAL_TEST, Y_TEST)]:
        stack = clf.predict_proba(meta_features(cal, names))
        blend = normp((1 - alpha) * cal[anchor] + alpha * stack)
        logits = np.log(np.clip(blend, 1e-12, None)) + bias[None, :]
        e = np.exp(logits - logits.max(1, keepdims=True))
        out[tag] = normp(e / e.sum(1, keepdims=True))
    return out

FULL_CLF = fit_stack(MODEL_NAMES)
FULL_OUT = apply_pipeline(FULL_CLF, MODEL_NAMES)
print("Refit-stack test BACC :", f"{100*full_metrics(Y_TEST, FULL_OUT['test'])['bacc']:.4f}%")
print("Frozen  stack test BACC:", f"{100*full_metrics(Y_TEST, FINAL_TEST)['bacc']:.4f}%")
print("(small differences are expected; the FROZEN arrays remain authoritative for reporting)")

# ---- global source importance from meta-classifier coefficients ----------
coef = FULL_CLF.named_steps["lr"].coef_          # (K, 18*S)
block = 2 * NUM_CLASSES + 2
imp = []
for i, n in enumerate(MODEL_NAMES):
    sl = slice(i * block, (i + 1) * block)
    imp.append(float(np.mean(np.abs(coef[:, sl]))))
imp = np.array(imp)
IMPORT_DF = pd.DataFrame({"source_model": MODEL_NAMES,
                          "mean_abs_meta_coefficient": imp,
                          "normalized_meta_contribution": 100 * imp / imp.sum()})

# ---- leave-one-source-out -------------------------------------------------
loso_rows = []
base_bacc = full_metrics(Y_TEST, FULL_OUT["test"])["bacc"]
base_pred = FULL_OUT["test"].argmax(1)
for n in MODEL_NAMES:
    keep = [m for m in MODEL_NAMES if m != n]
    clf = fit_stack(keep)
    out = apply_pipeline(clf, keep)
    fm_t = full_metrics(Y_TEST, out["test"]); fm_v = full_metrics(Y_VAL, out["val"])
    flips = int(np.sum(out["test"].argmax(1) != base_pred))
    loso_rows.append({
        "removed_source": n,
        "val_bacc_without": fm_v["bacc"],
        "test_bacc_without": fm_t["bacc"],
        "bacc_drop_when_removed": base_bacc - fm_t["bacc"],
        "prediction_flips": flips,
        "mean_probability_shift": float(np.mean(np.abs(out["test"] - FULL_OUT["test"]))),
    })
    print(f"  LOSO -{n:20s} test BACC {100*fm_t['bacc']:.3f}%  "
          f"(drop {100*(base_bacc-fm_t['bacc']):+.3f} pp, {flips} flips)")

LOSO_DF = pd.DataFrame(loso_rows)
SOURCE_DF = IMPORT_DF.merge(LOSO_DF, left_on="source_model", right_on="removed_source") \
                     .drop(columns=["removed_source"]) \
                     .sort_values("normalized_meta_contribution", ascending=False)
display(SOURCE_DF.round(4))
SOURCE_DF.to_csv(TABLE_DIR / "Table_source_importance_loso.csv", index=False)

top = SOURCE_DF.iloc[0]
worst = SOURCE_DF.sort_values("bacc_drop_when_removed", ascending=False).iloc[0]
record("source.top_contributor", str(top.source_model))
record("source.top_contribution_pct", float(top.normalized_meta_contribution))
record("source.min_contribution_pct", float(SOURCE_DF.normalized_meta_contribution.min()))
record("source.largest_loso_source", str(worst.source_model))
record("source.largest_loso_drop", float(worst.bacc_drop_when_removed))
record("source.largest_loso_flips", int(worst.prediction_flips))

# ---- Fig 6: importance + LOSO --------------------------------------------
fig, (a0, a1) = plt.subplots(1, 2, figsize=(10.4, 4.2))
o = SOURCE_DF.sort_values("normalized_meta_contribution")
a0.barh(o.source_model, o.normalized_meta_contribution, color="#4C72B0")
for i, v in enumerate(o.normalized_meta_contribution):
    a0.text(v + 0.2, i, f"{v:.2f}", va="center", fontsize=8)
a0.set_xlabel("Normalised meta-logit contribution (%)")
a0.set_title("a  Stacking source contribution", fontsize=10, loc="left"); a0.grid(axis="x", alpha=0.25)

o2 = SOURCE_DF.sort_values("bacc_drop_when_removed")
cols = ["#C44E52" if v > 0 else "#55A868" for v in o2.bacc_drop_when_removed]
a1.barh(o2.source_model, 100 * o2.bacc_drop_when_removed, color=cols)
a1.axvline(0, color="k", lw=0.8)
a1.set_xlabel("$\\Delta$ test BACC when removed (percentage points)")
a1.set_title("b  Leave-one-source-out", fontsize=10, loc="left"); a1.grid(axis="x", alpha=0.25)
plt.tight_layout(); plt.savefig(FIG_DIR / "Fig6_source_contribution.png"); plt.close()
print("Fig6_source_contribution.png written")

Refit-stack test BACC : 83.9549%
Frozen  stack test BACC: 83.9549%
(small differences are expected; the FROZEN arrays remain authoritative for reporting)
  LOSO -SimCLR_NoTTA         test BACC 83.624%  (drop +0.331 pp, 13 flips)
  LOSO -SimCLR_SelectedTTA   test BACC 83.642%  (drop +0.313 pp, 12 flips)
  LOSO -Rotation             test BACC 83.886%  (drop +0.069 pp, 73 flips)
  LOSO -FusionConcat         test BACC 83.455%  (drop +0.500 pp, 12 flips)
  LOSO -FusionGatedV2        test BACC 83.720%  (drop +0.235 pp, 17 flips)
  LOSO -FusionGatedV3        test BACC 84.413%  (drop -0.459 pp, 18 flips)
  LOSO -ResidualFusion       test BACC 84.011%  (drop -0.057 pp, 13 flips)


         source_model  mean_abs_meta_coefficient  normalized_meta_contribution  val_bacc_without  test_bacc_without  bacc_drop_when_removed  prediction_flips  mean_probability_shift
2            Rotation                     0.0635                       20.6119            0.8235             0.8389                  0.0007                73                  0.0083
1  SimCLR_SelectedTTA                     0.0444                       14.4046            0.8414             0.8364                  0.0031                12                  0.0022
0        SimCLR_NoTTA                     0.0443                       14.3584            0.8402             0.8362                  0.0033                13                  0.0022
4       FusionGatedV2                     0.0411                       13.3422            0.8399             0.8372                  0.0024                17                  0.0020
3        FusionConcat                     0.0396                       12.8525            

Fig6_source_contribution.png written


In [16]:
# ============================================================
# CELL 16 — PRUNED CONFIGURATION (greedy forward, VALIDATION only)
# ============================================================
# Protocol note: selection uses validation BACC exclusively. The locked test
# partition is scored once, after the pruned subset has been fixed.
MAX_SOURCES = 4
selected, history = [], []
remaining = list(MODEL_NAMES)

while len(selected) < MAX_SOURCES and remaining:
    best = None
    for cand in remaining:
        trial = selected + [cand]
        clf = fit_stack(trial)
        out = apply_pipeline(clf, trial)
        v = full_metrics(Y_VAL, out["val"])["bacc"]
        if best is None or v > best[1]:
            best = (cand, v)
    selected.append(best[0]); remaining.remove(best[0])
    clf = fit_stack(selected); out = apply_pipeline(clf, selected)
    history.append({"n_sources": len(selected), "added": best[0],
                    "sources": "+".join(selected),
                    "val_bacc": full_metrics(Y_VAL, out["val"])["bacc"],
                    "test_bacc": full_metrics(Y_TEST, out["test"])["bacc"],
                    "test_accuracy": full_metrics(Y_TEST, out["test"])["accuracy"],
                    "test_macro_f1": full_metrics(Y_TEST, out["test"])["macro_f1"]})
    print(f"  +{best[0]:20s} val BACC {100*best[1]:.3f}%  "
          f"test BACC {100*history[-1]['test_bacc']:.3f}%")

PRUNE_DF = pd.DataFrame(history)
display(PRUNE_DF.round(4))
PRUNE_DF.to_csv(TABLE_DIR / "Table_pruned_configuration_search.csv", index=False)

# The reported pruned model = best validation BACC among 2-4 source subsets.
best_row = PRUNE_DF.loc[PRUNE_DF.val_bacc.idxmax()]
PRUNED_SOURCES = best_row.sources.split("+")
record("pruned.sources", PRUNED_SOURCES)
record("pruned.n_sources", int(best_row.n_sources))
record("pruned.val_bacc", float(best_row.val_bacc))
record("pruned.test_bacc", float(best_row.test_bacc))
record("pruned.retained_fraction_of_gain",
       float((best_row.test_bacc - full_metrics(Y_TEST, TEST_PROB["SimCLR_NoTTA"])["bacc"]) /
             max(base_bacc - full_metrics(Y_TEST, TEST_PROB["SimCLR_NoTTA"])["bacc"], 1e-9)))
print("\nPruned configuration selected on validation:", PRUNED_SOURCES)
print("  val BACC :", f"{100*best_row.val_bacc:.3f}%")
print("  test BACC:", f"{100*best_row.test_bacc:.3f}%")

  +Rotation             val BACC 81.206%  test BACC 81.485%
  +FusionGatedV3        val BACC 83.961%  test BACC 83.309%
  +FusionConcat         val BACC 83.788%  test BACC 83.574%
  +FusionGatedV2        val BACC 84.145%  test BACC 83.326%


   n_sources          added                                            sources  val_bacc  test_bacc  test_accuracy  test_macro_f1
0          1       Rotation                                           Rotation    0.8121     0.8148         0.8311         0.7787
1          2  FusionGatedV3                             Rotation+FusionGatedV3    0.8396     0.8331         0.8335         0.7909
2          3   FusionConcat                Rotation+FusionGatedV3+FusionConcat    0.8379     0.8357         0.8331         0.7868
3          4  FusionGatedV2  Rotation+FusionGatedV3+FusionConcat+FusionGatedV2    0.8415     0.8333         0.8339         0.7886


Pruned configuration selected on validation: ['Rotation', 'FusionGatedV3', 'FusionConcat', 'FusionGatedV2']
  val BACC : 84.145%
  test BACC: 83.326%


## 7 — Computational cost (Table 7) — **GPU required**

Architectures are rebuilt from the published specification; parameters and FLOPs
do not depend on trained weights, and latency is measured on the live accelerator.
The stack does **not** need seven encoder passes: the four fusion heads consume
features already extracted by the SimCLR and Rotation encoders, so the true cost
is *encoder passes + head passes*. Both are reported.

In [17]:
# ============================================================
# CELL 17 — BUILD ARCHITECTURES FOR COST MEASUREMENT
# ============================================================
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass

GPUS = tf.config.list_physical_devices("GPU")
DEVICE_NAME = "CPU"
if GPUS:
    try:
        DEVICE_NAME = tf.config.experimental.get_device_details(GPUS[0]).get("device_name", "GPU")
    except Exception:
        DEVICE_NAME = "GPU"
print("TensorFlow:", tf.__version__, "| device:", DEVICE_NAME)
record("efficiency.device", DEVICE_NAME)

IMG = 224
FEAT_DIM = 2048

def resnet_branch(dropout=0.30, name="branch"):
    base = keras.applications.ResNet50(include_top=False, weights=None,
                                       input_shape=(IMG, IMG, 3), pooling="avg")
    x = layers.Dropout(dropout)(base.output)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(base.input, out, name=name)

def fusion_concat_head():
    s = keras.Input((FEAT_DIM,)); r = keras.Input((FEAT_DIM,))
    x = layers.Concatenate()([s, r])
    for u in (512, 256):
        x = layers.Dense(u)(x); x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x); x = layers.Dropout(0.40)(x)
    return keras.Model([s, r], layers.Dense(NUM_CLASSES, activation="softmax")(x),
                       name="FusionConcat_head")

def fusion_gated_v2_head():
    s = keras.Input((FEAT_DIM,)); r = keras.Input((FEAT_DIM,))
    ps = layers.Dense(256)(s); pr = layers.Dense(256)(r)
    g = layers.Dense(128, activation="relu")(layers.Concatenate()([s, r]))
    a = layers.Dense(2, activation="softmax")(g)
    z = layers.Add()([layers.Multiply()([ps, layers.Lambda(lambda t: t[:, :1])(a)]),
                      layers.Multiply()([pr, layers.Lambda(lambda t: t[:, 1:])(a)])])
    x = layers.Dropout(0.50)(layers.Dense(128, activation="relu")(z))
    return keras.Model([s, r], layers.Dense(NUM_CLASSES, activation="softmax")(x),
                       name="FusionGatedV2_head")

def fusion_gated_v3_head():
    s = keras.Input((FEAT_DIM,)); r = keras.Input((FEAT_DIM,))
    ps = layers.Dense(256)(s); pr = layers.Dense(256)(r)
    g = layers.Dense(128, activation="relu")(layers.Concatenate()([ps, pr]))
    a = layers.Dense(2, activation="softmax")(g)
    gated = layers.Add()([layers.Multiply()([ps, layers.Lambda(lambda t: t[:, :1])(a)]),
                          layers.Multiply()([pr, layers.Lambda(lambda t: t[:, 1:])(a)])])
    inter = layers.Concatenate()([gated, ps, pr,
                                  layers.Lambda(lambda t: tf.abs(t[0] - t[1]))([ps, pr]),
                                  layers.Multiply()([ps, pr])])
    x = layers.Dropout(0.50)(layers.Dense(256, activation="relu")(inter))
    x = layers.Dropout(0.40)(layers.Dense(128, activation="relu")(x))
    return keras.Model([s, r], layers.Dense(NUM_CLASSES, activation="softmax")(x),
                       name="FusionGatedV3_head")

def residual_fusion_head():
    s = keras.Input((FEAT_DIM,)); r = keras.Input((FEAT_DIM,))
    def proj(t):
        t = layers.LayerNormalization()(t); t = layers.Dense(256)(t)
        t = layers.LayerNormalization()(t); return layers.Activation("relu")(t)
    ps, pr = proj(s), proj(r)
    inter = layers.Concatenate()([ps, pr,
                                  layers.Lambda(lambda t: tf.abs(t[0] - t[1]))([ps, pr]),
                                  layers.Multiply()([ps, pr])])
    x = layers.Dropout(0.40)(layers.Dense(256, activation="relu")(inter))
    return keras.Model([s, r], layers.Dense(NUM_CLASSES)(x), name="ResidualFusion_head")

ARCH = OrderedDict([
    ("ResNet50 encoder + head (SimCLR / Rotation / Supervised)", resnet_branch()),
    ("FusionConcat head",   fusion_concat_head()),
    ("FusionGatedV2 head",  fusion_gated_v2_head()),
    ("FusionGatedV3 head",  fusion_gated_v3_head()),
    ("ResidualFusion head", residual_fusion_head()),
])
for k, m in ARCH.items():
    print(f"{k:58s} params = {m.count_params():,}")

TensorFlow: 2.20.0 | device: Tesla T4


I0000 00:00:1788845612.114465      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788845612.117365      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


ResNet50 encoder + head (SimCLR / Rotation / Supervised)   params = 23,604,104
FusionConcat head                                          params = 2,234,120
FusionGatedV2 head                                         params = 1,607,690
FusionGatedV3 head                                         params = 1,476,874
ResidualFusion head                                        params = 1,322,760


In [18]:
# ============================================================
# CELL 18 — MEASURE PARAMS / FLOPs / LATENCY / MEMORY
# ============================================================
def count_flops(model, input_shapes):
    """Forward FLOPs via the TF profiler. Returns MACs (FLOPs/2) or NaN."""
    try:
        from tensorflow.python.profiler.model_analyzer import profile
        from tensorflow.python.profiler.option_builder import ProfileOptionBuilder
        specs = [tf.TensorSpec([1] + list(s), tf.float32) for s in input_shapes]
        cf = tf.function(lambda *a: model(a if len(a) > 1 else a[0])).get_concrete_function(*specs)
        graph = cf.graph.as_graph_def()
        with tf.Graph().as_default() as g:
            tf.graph_util.import_graph_def(graph, name="")
            opts = ProfileOptionBuilder(ProfileOptionBuilder.float_operation()).with_empty_output().build()
            flops = profile(g, options=opts)
        return float(flops.total_float_ops) / 2.0
    except Exception as exc:
        print("   FLOPs profiler unavailable:", exc)
        return float("nan")

def measure_latency(model, input_shapes, warmup=LATENCY_WARMUP, repeats=LATENCY_REPEATS):
    xs = [tf.random.normal([1] + list(s)) for s in input_shapes]
    fn = tf.function(lambda *a: model(a if len(a) > 1 else a[0], training=False))
    for _ in range(warmup):
        _ = fn(*xs)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter(); _ = fn(*xs)
        try: tf.experimental.numpy.empty(0)
        except Exception: pass
        times.append((time.perf_counter() - t0) * 1000.0)
    times = np.array(times)
    return float(np.median(times)), float(times.std())

def peak_memory_mb():
    try:
        return tf.config.experimental.get_memory_info("GPU:0")["peak"] / (1024 ** 2)
    except Exception:
        return float("nan")

SPEC = {
    "ResNet50 encoder + head (SimCLR / Rotation / Supervised)": [(IMG, IMG, 3)],
    "FusionConcat head":   [(FEAT_DIM,), (FEAT_DIM,)],
    "FusionGatedV2 head":  [(FEAT_DIM,), (FEAT_DIM,)],
    "FusionGatedV3 head":  [(FEAT_DIM,), (FEAT_DIM,)],
    "ResidualFusion head": [(FEAT_DIM,), (FEAT_DIM,)],
}

comp_rows = []
for name, model in ARCH.items():
    shapes = SPEC[name]
    try: tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception: pass
    macs = count_flops(model, shapes)
    lat, sd = measure_latency(model, shapes)
    comp_rows.append({"component": name, "params": int(model.count_params()),
                      "params_M": model.count_params() / 1e6,
                      "gmac": macs / 1e9 if macs == macs else float("nan"),
                      "latency_ms_median": lat, "latency_ms_sd": sd,
                      "peak_memory_mb": peak_memory_mb()})
    print(f"{name:58s} {lat:7.2f} ms  {comp_rows[-1]['gmac']:.3f} GMAC")

COMPONENT_DF = pd.DataFrame(comp_rows)
display(COMPONENT_DF.round(4))
COMPONENT_DF.to_csv(TABLE_DIR / "Table7A_component_cost.csv", index=False)

def comp(name, field):
    return float(COMPONENT_DF.loc[COMPONENT_DF.component == name, field].iloc[0])

ENC = "ResNet50 encoder + head (SimCLR / Rotation / Supervised)"
enc_p, enc_g, enc_l = comp(ENC, "params_M"), comp(ENC, "gmac"), comp(ENC, "latency_ms_median")
head = {n: (comp(n, "params_M"), comp(n, "gmac"), comp(n, "latency_ms_median"))
        for n in ["FusionConcat head", "FusionGatedV2 head",
                  "FusionGatedV3 head", "ResidualFusion head"]}
HEAD_ALL = tuple(sum(v[i] for v in head.values()) for i in range(3))

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


233 ops no flops stats due to incomplete shapes.


ResNet50 encoder + head (SimCLR / Rotation / Supervised)      7.94 ms  0.000 GMAC
FusionConcat head                                             1.07 ms  0.000 GMAC


21 ops no flops stats due to incomplete shapes.
17 ops no flops stats due to incomplete shapes.


FusionGatedV2 head                                            1.05 ms  0.000 GMAC
FusionGatedV3 head                                            1.19 ms  0.000 GMAC


21 ops no flops stats due to incomplete shapes.
14 ops no flops stats due to incomplete shapes.


ResidualFusion head                                           1.29 ms  0.000 GMAC


                                           component    params  params_M    gmac  latency_ms_median  latency_ms_sd  peak_memory_mb
0  ResNet50 encoder + head (SimCLR / Rotation / S...  23604104   23.6041  0.0001             7.9434         2.5840       1261.1741
1                                  FusionConcat head   2234120    2.2341  0.0000             1.0720         0.0758        123.4355
2                                 FusionGatedV2 head   1607690    1.6077  0.0000             1.0543         0.0664        123.4355
3                                 FusionGatedV3 head   1476874    1.4769  0.0000             1.1864         0.0568        123.4355
4                                ResidualFusion head   1322760    1.3228  0.0000             1.2912         0.0530        123.4517

In [19]:
# ============================================================
# CELL 19 — TABLE 7: CONFIGURATION-LEVEL COST
# ============================================================
# Seed counts as deployed in the frozen pipeline.
SEEDS = {"SimCLR": 5, "Rotation": 1, "Supervised": 5, "ResidualFusion": 3}

def build_config(label, enc_passes, heads, bacc):
    hp = sum(head[h][0] for h in heads)
    hg = sum(head[h][1] for h in heads)
    hl = sum(head[h][2] for h in heads)
    return {"configuration": label,
            "encoder_passes": enc_passes,
            "params_M": enc_passes * enc_p + hp,
            "gmac": enc_passes * enc_g + hg,
            "latency_ms": enc_passes * enc_l + hl,
            "bacc": bacc}

def bacc_of(m):
    return float(TABLE4_RAW.loc[TABLE4_RAW.Model == m, "bacc"].iloc[0]) \
        if m in set(TABLE4_RAW.Model) else float("nan")

cfgs = [
    build_config("Rotation (single branch)", 1, [], bacc_of("Rotation")),
    build_config("SimCLR No-TTA (1 seed)",   1, [], float("nan")),
    build_config("SimCLR No-TTA (5 seeds, as deployed)", SEEDS["SimCLR"], [],
                 bacc_of("SimCLR No-TTA")),
    build_config("Supervised ResNet50 (5 seeds)", SEEDS["Supervised"], [],
                 bacc_of("Supervised ResNet50")),
]

# Pruned configuration: encoder passes implied by the selected sources.
def encoder_passes_for(sources):
    n = 0
    if any(s.startswith("SimCLR_NoTTA") for s in sources):        n += SEEDS["SimCLR"]
    if any(s.startswith("SimCLR_SelectedTTA") for s in sources):  n += SEEDS["SimCLR"]  # extra hflip views
    if "Rotation" in sources:                                     n += SEEDS["Rotation"]
    if n == 0 and sources:                                        n = SEEDS["SimCLR"] + SEEDS["Rotation"]
    if any(s.startswith("Fusion") or s == "ResidualFusion" for s in sources):
        need = SEEDS["SimCLR"] + SEEDS["Rotation"]
        n = max(n, need)
    return n

HEAD_MAP = {"FusionConcat": "FusionConcat head", "FusionGatedV2": "FusionGatedV2 head",
            "FusionGatedV3": "FusionGatedV3 head", "ResidualFusion": "ResidualFusion head"}

pruned_heads = [HEAD_MAP[s] for s in PRUNED_SOURCES if s in HEAD_MAP]
cfgs.append(build_config(
    f"Pruned CalStack-SSL ({len(PRUNED_SOURCES)} sources)",
    encoder_passes_for(PRUNED_SOURCES), pruned_heads,
    float(record("pruned.test_bacc_check", MANUSCRIPT["pruned.test_bacc"]["value"]))))

full_heads = [HEAD_MAP[s] for s in MODEL_NAMES if s in HEAD_MAP]
cfgs.append(build_config("CalStack-SSL (7 sources)",
                         encoder_passes_for(MODEL_NAMES), full_heads,
                         bacc_of("CalStack-SSL")))

TABLE7 = pd.DataFrame(cfgs)
TABLE7["bacc_pct"] = 100 * TABLE7["bacc"]
display(TABLE7.round(3))
TABLE7.to_csv(TABLE_DIR / "Table7_configuration_cost.csv", index=False)

full_row   = TABLE7.iloc[-1]
pruned_row = TABLE7.iloc[-2]
record("efficiency.full_gmac", float(full_row.gmac))
record("efficiency.full_latency_ms", float(full_row.latency_ms))
record("efficiency.full_params_M", float(full_row.params_M))
record("efficiency.full_encoder_passes", int(full_row.encoder_passes))
record("efficiency.pruned_gmac", float(pruned_row.gmac))
record("efficiency.pruned_latency_ms", float(pruned_row.latency_ms))
record("efficiency.pruned_cost_fraction_pct",
       float(100 * pruned_row.latency_ms / full_row.latency_ms))
record("efficiency.single_simclr_gmac", float(TABLE7.iloc[2].gmac))
record("efficiency.single_simclr_latency_ms", float(TABLE7.iloc[2].latency_ms))
record("efficiency.latency_repeats", LATENCY_REPEATS)
record("efficiency.latency_warmup", LATENCY_WARMUP)

del ARCH; gc.collect(); keras.backend.clear_session()

                          configuration  encoder_passes  params_M   gmac  latency_ms   bacc  bacc_pct
0              Rotation (single branch)               1    23.604  0.000       7.943  0.737    73.744
1                SimCLR No-TTA (1 seed)               1    23.604  0.000       7.943    NaN       NaN
2  SimCLR No-TTA (5 seeds, as deployed)               5   118.021  0.000      39.717  0.811    81.073
3         Supervised ResNet50 (5 seeds)               5   118.021  0.000      39.717  0.802    80.170
4       Pruned CalStack-SSL (4 sources)               6   146.943  0.000      50.973  0.833    83.326
5              CalStack-SSL (7 sources)              11   266.287  0.001      91.981  0.840    83.955

## 8 — Confusion matrices, external results, Grad-CAM figures

In [20]:
# ============================================================
# CELL 20 — Fig 8: CONFUSION MATRICES
# ============================================================
def plot_cm(ax, y, p, labels, title):
    cm = confusion_matrix(y, np.argmax(p, 1), labels=np.arange(len(labels)))
    cmn = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    im = ax.imshow(cmn, vmin=0, vmax=1, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels)), labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            if cm[i, j] > 0:
                ax.text(j, i, f"{100*cmn[i,j]:.0f}", ha="center", va="center", fontsize=7,
                        color="white" if cmn[i, j] > 0.55 else "black")
    ax.set_title(title, fontsize=10, loc="left")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    return im

fig, axes = plt.subplots(1, 1, figsize=(5.2, 4.6))
plot_cm(axes, Y_TEST, FINAL_TEST, CLASS_NAMES, "a  Locked ISIC 2019 test")
plt.tight_layout(); plt.savefig(FIG_DIR / "Fig8_confusion_matrices.png"); plt.close()
print("Fig8_confusion_matrices.png written (locked-test panel)")
print("NOTE: add official-ISIC and PAD panels by re-running with those probability")
print("      arrays attached; the helper plot_cm() takes any (y, p, labels).")

pd.DataFrame(confusion_matrix(Y_TEST, FINAL_TEST.argmax(1), labels=np.arange(NUM_CLASSES)),
             index=[f"true_{c}" for c in CLASS_NAMES],
             columns=[f"pred_{c}" for c in CLASS_NAMES]) \
  .to_csv(TABLE_DIR / "confusion_matrix_locked_test.csv")

Fig8_confusion_matrices.png written (locked-test panel)
NOTE: add official-ISIC and PAD panels by re-running with those probability
      arrays attached; the helper plot_cm() takes any (y, p, labels).


In [21]:
# ============================================================
# CELL 21 — EXTERNAL / DOMAIN-SHIFT TABLES (reuse frozen results)
# ============================================================
EXT_FRAMES = {}
for key, label in [("external_results", "external_eight_class"),
                   ("pad_results", "pad_all_adapted"),
                   ("pad_da_results", "pad_domain_adapted_stacking")]:
    if ART[key]:
        df = pd.read_csv(ART[key])
        EXT_FRAMES[label] = df
        df.to_csv(TABLE_DIR / f"Table8_{label}.csv", index=False)
        print(f"--- {label}  ({ART[key]})")
        display(df.head(20))
    else:
        print(f"!! {label} not found — attach the producing notebook output.")

json.dump({k: list(v.columns) for k, v in EXT_FRAMES.items()},
          open(AUDIT_DIR / "external_table_columns.json", "w"), indent=2)

--- external_eight_class  (/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/both_stacking_models_external_evaluation/all_external_eight_class_results.csv)


                    dataset                         method  n_total  n_labeled  accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f1  macro_specificity  auc_macro_ovr
0   ISIC_2019_Official_Test                   SimCLR_NoTTA     8238       6191  0.684219  0.516550         0.583740      0.516550  0.541321     0.680087           0.949278       0.911842
1   ISIC_2019_Official_Test             SimCLR_SelectedTTA     8238       6191  0.684542  0.515507         0.590444      0.515507  0.542474     0.680321           0.949269       0.913045
2   ISIC_2019_Official_Test                       Rotation     8238       6191  0.670005  0.534585         0.579538      0.534585  0.549291     0.663552           0.945600       0.907161
3   ISIC_2019_Official_Test                   FusionConcat     8238       6191  0.650299  0.506839         0.525701      0.506839  0.493482     0.649839           0.945772       0.861438
4   ISIC_2019_Official_Test                  FusionGatedV2     82

!! pad_all_adapted not found — attach the producing notebook output.
--- pad_domain_adapted_stacking  (/kaggle/input/notebooks/anirbanroysudipto/confidence-gated-ensemble/pad_domain_adapted_both_stacking_models/PAD_DOMAIN_ADAPTED_BOTH_STACKING_RESULTS.csv)


                                        method                                        result_type  n_samples  accuracy      bacc  macro_precision  macro_recall  macro_f1  weighted_f1  auc_macro_ovr  \
0  Previous_PAD_FineTuned_FusionConcat_Adapter  Previous PAD fine-tuned/domain-adapted test re...        345  0.689855  0.608252         0.653708      0.608252  0.616420     0.678759       0.902534   
1                   PAD_DA_NoTTA_StackingBlend                 PAD fine-tuned/domain-adapted test        345  0.527536  0.547127         0.450807      0.547127  0.461130     0.532081       0.847375   
2         PAD_DA_NoTTA_StackingBlend_ClassBias                 PAD fine-tuned/domain-adapted test        345  0.533333  0.543817         0.450081      0.543817  0.458668     0.531402       0.846608   
3       ZeroShot_NoTTA_StackingBlend_ClassBias               Zero-shot on PAD adapted-test subset        345  0.307246  0.351670         0.345502      0.351670  0.265098     0.304673       0.74191

In [22]:
# ============================================================
# CELL 22 — COPY / STITCH EXISTING FIGURES (Grad-CAM, framework, ESM)
# ============================================================
COPY_MAP = {
    "gradcam_correct_per_class.png":                    FIG_DIR / "Fig7_gradcam.png",
    "gradcam_final_rescue_cases.png":                   ESM_DIR / "FigS2_gradcam_rescue.png",
    "simclr_validation_umap.png":                       ESM_DIR / "FigS3_umap_simclr.png",
    "rotation_validation_umap.png":                     ESM_DIR / "FigS4_umap_rotation.png",
    "dinov2_validation_umap.png":                       ESM_DIR / "FigS5_umap_dinov2.png",
    "classwise_meta_source_importance.png":             ESM_DIR / "FigS6_classwise_importance.png",
    "case_level_source_contribution_heatmap.png":       ESM_DIR / "FigS7_case_contribution.png",
    "progressive_ablation_test_bacc.png":               ESM_DIR / "FigS8_progressive_ablation.png",
    "ensemble_progression_bacc.png":                    ESM_DIR / "FigS9_ensemble_progression.png",
}
copy_rows = []
for src_name, dst in COPY_MAP.items():
    hit = find_one(src_name, ("j7", "j8", "j5", "j3"))
    if hit:
        shutil.copy2(hit, dst)
    copy_rows.append({"figure": src_name, "copied": hit is not None,
                      "destination": dst.name, "source": str(hit) if hit else ""})
CP_DF = pd.DataFrame(copy_rows)
display(CP_DF[["figure", "copied", "destination"]])
CP_DF.to_csv(AUDIT_DIR / "figure_copy_manifest.csv", index=False)

missing_main = [r["figure"] for _, r in CP_DF.iterrows()
                if not r["copied"] and "Fig7" in r["destination"]]
if missing_main:
    print("\n!! Fig7 (Grad-CAM) not found. Attach journal-extension-j7 or j8 output,")
    print("   or regenerate the panel from the SimCLR/Rotation weights.")

print("\nManuscript still needs, supplied manually:")
print("  Fig1_framework.png            (draw: pipeline schematic)")
print("  Fig2_ensemble_construction.png(draw: decision-layer schematic)")
print("  Graphical_Abstract_Roy.tif    (MANDATORY, 32.93 x 37.63 mm, >=600 dpi)")

                                       figure  copied                     destination
0               gradcam_correct_per_class.png    True                Fig7_gradcam.png
1              gradcam_final_rescue_cases.png    True        FigS2_gradcam_rescue.png
2                  simclr_validation_umap.png    True           FigS3_umap_simclr.png
3                rotation_validation_umap.png    True         FigS4_umap_rotation.png
4                  dinov2_validation_umap.png    True           FigS5_umap_dinov2.png
5        classwise_meta_source_importance.png    True  FigS6_classwise_importance.png
6  case_level_source_contribution_heatmap.png    True     FigS7_case_contribution.png
7          progressive_ablation_test_bacc.png    True  FigS8_progressive_ablation.png
8               ensemble_progression_bacc.png    True  FigS9_ensemble_progression.png


Manuscript still needs, supplied manually:
  Fig1_framework.png            (draw: pipeline schematic)
  Fig2_ensemble_construction.png(draw: decision-layer schematic)
  Graphical_Abstract_Roy.tif    (MANDATORY, 32.93 x 37.63 mm, >=600 dpi)


## 9 — Emit paste-ready LaTeX table bodies and the master value file

In [23]:
# ============================================================
# CELL 23 — LATEX: Table 4 (main results with CIs)
# ============================================================
ORDER4 = ["Supervised ResNet50", "SimCLR No-TTA", "SimCLR selected-TTA",
          "DINOv2 ViT-S/14", "Rotation", "FusionConcat", "FusionGated V2",
          "FusionGated V3", "Residual Fusion", "Weighted-average ensemble",
          "Anchor + stacking (no bias)", "CalStack-SSL"]
LABEL4 = {"Supervised ResNet50": "Supervised ResNet50 (5 seeds)",
          "SimCLR No-TTA": "SimCLR No-TTA (5 seeds)",
          "SimCLR selected-TTA": "SimCLR selected-TTA (5 seeds)",
          "DINOv2 ViT-S/14": "DINOv2 ViT-S/14 (3 seeds)",
          "CalStack-SSL": r"\textbf{CalStack-SSL (proposed)}"}

best = {c: TABLE4_RAW[c].max() for c in
        ["accuracy", "bacc", "macro_f1", "macro_specificity", "auc_macro_ovr", "mcc"]}

lines = []
for m in ORDER4:
    if m not in set(TABLE4_RAW.Model):
        continue
    r = TABLE4_RAW.loc[TABLE4_RAW.Model == m].iloc[0]
    lo, hi = ci_of(m, "bacc")
    def cell(col, nd=2):
        v = float(r[col])
        s = f"{100*v:.{nd}f}"
        return r"\textbf{" + s + "}" if abs(v - best[col]) < 1e-12 else s
    if m == "CalStack-SSL":
        lines.append(r"\midrule")
    lines.append(
        f"{LABEL4.get(m, m)} & {cell('accuracy')} & {cell('bacc')} {fmt_ci(lo, hi)} & "
        f"{cell('macro_f1')} & {cell('macro_specificity')} & {cell('auc_macro_ovr')} & "
        f"{cell('mcc')} \\\\"
    )

TABLE4_TEX = "\n".join(lines)
print(TABLE4_TEX)
write_latex("Table4_body.tex", TABLE4_TEX,
            "Columns: Method & Acc & BACC (95% CI) & Macro-F1 & Macro-spec & Macro-AUC & MCC")

Supervised ResNet50 (5 seeds) & 84.85 & 80.17 (77.08--83.07) & \textbf{81.48} & 97.20 & 97.60 & 77.53 \\
SimCLR No-TTA (5 seeds) & 84.45 & 81.07 (77.98--83.93) & 81.44 & 97.26 & 97.79 & 77.23 \\
SimCLR selected-TTA (5 seeds) & 84.57 & 80.59 (77.44--83.49) & 80.69 & 97.30 & 97.80 & 77.42 \\
DINOv2 ViT-S/14 (3 seeds) & 79.64 & 80.43 (77.51--83.18) & 78.83 & 96.63 & 97.40 & 71.14 \\
Rotation & 79.52 & 73.74 (70.25--77.08) & 73.73 & 96.31 & 96.58 & 69.76 \\
FusionConcat & 82.28 & 78.73 (75.55--81.86) & 75.05 & 96.96 & 96.14 & 74.27 \\
FusionGated V2 & 82.56 & 77.22 (73.85--80.55) & 76.60 & 96.90 & 96.77 & 74.36 \\
FusionGated V3 & 82.32 & 77.43 (74.14--80.58) & 76.62 & 96.84 & 96.41 & 73.95 \\
Residual Fusion & 83.07 & 78.92 (75.78--82.05) & 78.47 & 96.98 & 96.80 & 75.11 \\
Weighted-average ensemble & \textbf{85.08} & 81.03 (77.90--84.03) & 81.29 & \textbf{97.31} & \textbf{97.92} & \textbf{78.03} \\
Anchor + stacking (no bias) & 83.70 & \textbf{84.00} (81.21--86.57) & 79.88 & 97.26 & 97.67

PosixPath('/kaggle/working/j9_mbec_results/latex/Table4_body.tex')

In [24]:
# ============================================================
# CELL 24 — LATEX: Table 5 (paired comparisons)
# ============================================================
ORDER5 = ["SimCLR No-TTA", "DINOv2 ViT-S/14", "Rotation",
          "Weighted-average ensemble", "Supervised ResNet50"]
lines = []
for comp in ORDER5:
    if comp not in COMPARATORS:
        continue
    rb, ra = paired_of(comp, "bacc"), paired_of(comp, "accuracy")
    mc = mcnemar_of(comp)
    lines.append(
        f"{comp} & "
        f"${fmt_signed(rb.observed_difference)}$ "
        f"(${fmt_signed(rb.ci95_low)}$, ${fmt_signed(rb.ci95_high)}$) & "
        f"${fmt_signed(ra.observed_difference)}$ "
        f"(${fmt_signed(ra.ci95_low)}$, ${fmt_signed(ra.ci95_high)}$) & "
        f"{fmt_p(float(mc.holm_adjusted_p))} \\\\"
    )
TABLE5_TEX = "\n".join(lines)
print(TABLE5_TEX)
write_latex("Table5_body.tex", TABLE5_TEX,
            "Columns: Comparator & dBACC (95% CI) & dAccuracy (95% CI) & McNemar p (Holm)")

SimCLR No-TTA & $+2.88$ ($+0.79$, $+5.22$) & $$-$0.55$ ($$-$1.38$, $+0.32$) & 0.298 \\
DINOv2 ViT-S/14 & $+3.52$ ($+0.85$, $+6.24$) & $+4.26$ ($+2.76$, $+5.80$) & $<$0.001 \\
Rotation & $+10.21$ ($+7.00$, $+13.59$) & $+4.38$ ($+2.96$, $+5.80$) & $<$0.001 \\
Weighted-average ensemble & $+2.93$ ($+1.00$, $+5.19$) & $$-$1.18$ ($$-$2.05$, $$-$0.36$) & 0.02 \\
Supervised ResNet50 & $+3.78$ ($+1.14$, $+6.52$) & $$-$0.95$ ($$-$2.21$, $+0.28$) & 0.298 \\
  wrote Table5_body.tex


PosixPath('/kaggle/working/j9_mbec_results/latex/Table5_body.tex')

In [25]:
# ============================================================
# CELL 25 — LATEX: Table 6 (per-class recall) and Table 7 (cost)
# ============================================================
cols6 = [m for m in ["Supervised ResNet50", "SimCLR No-TTA", "Rotation", "CalStack-SSL"]
         if m in TABLE6.columns]
lines = []
for _, r in TABLE6.iterrows():
    if r["Class"] == "Macro (BACC)":
        lines.append(r"\midrule")
    vals = []
    for m in cols6:
        s = f"{100*float(r[m]):.2f}"
        vals.append(r"\textbf{" + s + "}" if (r["Class"] == "Macro (BACC)"
                                             and m == "CalStack-SSL") else s)
    lines.append(f"{r['Class']} & {int(r['n'])} & " + " & ".join(vals) + r" \\")
TABLE6_TEX = "\n".join(lines)
print(TABLE6_TEX)
write_latex("Table6_body.tex", TABLE6_TEX,
            "Columns: Class & n & " + " & ".join(cols6))

lines = []
for _, r in TABLE7.iterrows():
    g = "--" if r.gmac != r.gmac else f"{r.gmac:.1f}"
    b = "--" if r.bacc != r.bacc else f"{100*r.bacc:.2f}"
    lines.append(f"{r.configuration} & {int(r.encoder_passes)} & {r.params_M:.1f} & "
                 f"{g} & {r.latency_ms:.1f} & {b} \\\\")
TABLE7_TEX = "\n".join(lines)
print("\n" + TABLE7_TEX)
write_latex("Table7_body.tex", TABLE7_TEX,
            "Columns: Configuration & Encoder passes & Params (M) & GMAC & Latency (ms) & BACC (%)")

MEL & 452 & 72.35 & 76.11 & 66.59 & 74.78 \\
NV & 1288 & 92.39 & 89.60 & 88.20 & 88.04 \\
BCC & 333 & 89.49 & 86.79 & 81.38 & 86.79 \\
AK & 86 & 69.77 & 76.74 & 61.63 & 79.07 \\
BKL & 263 & 70.34 & 74.90 & 65.40 & 74.90 \\
DF & 24 & 70.83 & 66.67 & 62.50 & 79.17 \\
VASC & 25 & 100.00 & 100.00 & 96.00 & 100.00 \\
SCC & 63 & 76.19 & 77.78 & 68.25 & 88.89 \\
\midrule
Macro (BACC) & 2534 & 80.17 & 81.07 & 73.74 & \textbf{83.95} \\
MCC & 2534 & 77.53 & 77.23 & 69.76 & 76.74 \\
  wrote Table6_body.tex

Rotation (single branch) & 1 & 23.6 & 0.0 & 7.9 & 73.74 \\
SimCLR No-TTA (1 seed) & 1 & 23.6 & 0.0 & 7.9 & -- \\
SimCLR No-TTA (5 seeds, as deployed) & 5 & 118.0 & 0.0 & 39.7 & 81.07 \\
Supervised ResNet50 (5 seeds) & 5 & 118.0 & 0.0 & 39.7 & 80.17 \\
Pruned CalStack-SSL (4 sources) & 6 & 146.9 & 0.0 & 51.0 & 83.33 \\
CalStack-SSL (7 sources) & 11 & 266.3 & 0.0 & 92.0 & 83.95 \\
  wrote Table7_body.tex


PosixPath('/kaggle/working/j9_mbec_results/latex/Table7_body.tex')

In [26]:
# ============================================================
# CELL 26 — MANUSCRIPT VALUE FILE + SENTENCE SNIPPETS
# ============================================================
def pct(key, nd=2):
    v = MANUSCRIPT[key]["value"]
    return f"{100*float(v):.{nd}f}"

snip = []
snip.append("=" * 90)
snip.append("PASTE-READY VALUES FOR THE MANUSCRIPT (replace each \\TBD{})")
snip.append("=" * 90)

snip.append("\n[Abstract / Section 4.1]")
snip.append(f"  CalStack-SSL BACC        : {pct('table4.calstack_bacc')}% "
            f"({fmt_ci(*ci_of('CalStack-SSL','bacc'))})")
snip.append(f"  Accuracy / Macro-F1 / MCC: {pct('table4.calstack_accuracy')}% / "
            f"{pct('table4.calstack_macro_f1')}% / {pct('table4.calstack_mcc')}%")
if "abstract.delta_bacc_vs_supervised" in MANUSCRIPT:
    snip.append(f"  Delta BACC vs supervised : {pct('abstract.delta_bacc_vs_supervised')} pp")

snip.append("\n[Section 4.2 / Table 5]")
for comp in COMPARATORS:
    rb = paired_of(comp, "bacc")
    snip.append(f"  vs {comp:28s} dBACC {100*rb.observed_difference:+.2f} pp "
                f"(95% CI {100*rb.ci95_low:+.2f}, {100*rb.ci95_high:+.2f}), "
                f"McNemar p={float(mcnemar_of(comp).holm_adjusted_p):.4g}")

snip.append("\n[Section 4.3]")
for cls in ["DF", "SCC", "MEL", "NV"]:
    k = f"classwise.delta_recall_vs_simclr.{cls}"
    if k in MANUSCRIPT:
        snip.append(f"  {cls} recall change vs SimCLR: {100*MANUSCRIPT[k]['value']:+.2f} pp")

snip.append("\n[Section 4.4]")
snip.append(f"  T* = {MANUSCRIPT['calibration.temperature']['value']:.3f}; "
            f"ECE {pct('calibration.ece_raw')}% -> {pct('calibration.ece_scaled')}%; "
            f"labels changed = {MANUSCRIPT['calibration.labels_changed']['value']}")
snip.append(f"  Accuracy @90% coverage: {pct('selective.acc_at_90')}%; "
            f"@50%: {pct('selective.acc_at_50')}%, BACC {pct('selective.bacc_at_50')}%")

snip.append("\n[Section 4.5]")
snip.append(f"  Top source: {MANUSCRIPT['source.top_contributor']['value']} at "
            f"{MANUSCRIPT['source.top_contribution_pct']['value']:.2f}%; "
            f"others >= {MANUSCRIPT['source.min_contribution_pct']['value']:.2f}%")
snip.append(f"  Largest LOSO: {MANUSCRIPT['source.largest_loso_source']['value']}, "
            f"-{100*MANUSCRIPT['source.largest_loso_drop']['value']:.2f} pp, "
            f"{MANUSCRIPT['source.largest_loso_flips']['value']} flips")
if "rescue.anchor_errors_rescued" in MANUSCRIPT:
    snip.append(f"  Anchor errors rescued: {MANUSCRIPT['rescue.anchor_errors_rescued']['value']} "
                f"(broken {MANUSCRIPT['rescue.anchor_correct_broken']['value']}, "
                f"net {MANUSCRIPT['rescue.net_gain']['value']:+d})")

snip.append("\n[Section 4.6 / Table 7]")
snip.append(f"  Device: {MANUSCRIPT['efficiency.device']['value']}")
snip.append(f"  Full stack : {MANUSCRIPT['efficiency.full_encoder_passes']['value']} encoder passes, "
            f"{MANUSCRIPT['efficiency.full_gmac']['value']:.1f} GMAC, "
            f"{MANUSCRIPT['efficiency.full_latency_ms']['value']:.1f} ms")
snip.append(f"  Pruned     : {MANUSCRIPT['pruned.sources']['value']}, "
            f"BACC {pct('pruned.test_bacc')}%, "
            f"{MANUSCRIPT['efficiency.pruned_cost_fraction_pct']['value']:.0f}% of full cost")
snip.append(f"  Retained fraction of full-stack gain: "
            f"{100*MANUSCRIPT['pruned.retained_fraction_of_gain']['value']:.0f}%")

TEXT = "\n".join(snip)
print(TEXT)
(ROOT / "MANUSCRIPT_SNIPPETS.txt").write_text(TEXT, encoding="utf-8")

json.dump(MANUSCRIPT, open(ROOT / "MANUSCRIPT_VALUES.json", "w"), indent=2, default=float)
print("\nWrote MANUSCRIPT_VALUES.json with", len(MANUSCRIPT), "keys.")

PASTE-READY VALUES FOR THE MANUSCRIPT (replace each \TBD{})

[Abstract / Section 4.1]
  CalStack-SSL BACC        : 83.95% ((81.20--86.52))
  Accuracy / Macro-F1 / MCC: 83.90% / 79.80% / 76.74%
  Delta BACC vs supervised : 3.78 pp

[Section 4.2 / Table 5]
  vs Supervised ResNet50          dBACC +3.78 pp (95% CI +1.14, +6.52), McNemar p=0.2977
  vs SimCLR No-TTA                dBACC +2.88 pp (95% CI +0.79, +5.22), McNemar p=0.2977
  vs DINOv2 ViT-S/14              dBACC +3.52 pp (95% CI +0.85, +6.24), McNemar p=4.612e-07
  vs Rotation                     dBACC +10.21 pp (95% CI +7.00, +13.59), McNemar p=1.166e-08
  vs Weighted-average ensemble    dBACC +2.93 pp (95% CI +1.00, +5.19), McNemar p=0.02049

[Section 4.3]
  DF recall change vs SimCLR: +12.50 pp
  SCC recall change vs SimCLR: +11.11 pp
  MEL recall change vs SimCLR: -1.33 pp
  NV recall change vs SimCLR: -1.55 pp

[Section 4.4]
  T* = 0.866; ECE 3.72% -> 1.93%; labels changed = 0
  Accuracy @90% coverage: 88.16%; @50%: 97.47%, 

In [27]:
# ============================================================
# CELL 27 — PACKAGE EVERYTHING
# ============================================================
REPORT = {
    "notebook": "J9_MBEC_manuscript_results",
    "protocol": "locked stratified 80/10/10 ISIC 2019",
    "test_n": int(len(Y_TEST)), "val_n": int(len(Y_VAL)),
    "bootstrap_replicates": N_BOOTSTRAP, "bootstrap_seed": BOOTSTRAP_SEED,
    "sources": MODEL_NAMES,
    "pruned_sources": MANUSCRIPT["pruned.sources"]["value"],
    "device": MANUSCRIPT["efficiency.device"]["value"],
    "test_used_for_selection": False,
    "n_manuscript_values": len(MANUSCRIPT),
}
json.dump(REPORT, open(ROOT / "J9_COMPLETE.json", "w"), indent=2)

ZIP_PATH = WORKING / "J9_MBEC_RESULTS_PACKAGE.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(ROOT.rglob("*")):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(ROOT.parent)))

REQUIRED = [
    TABLE_DIR / "Table4_locked_test_metrics.csv",
    TABLE_DIR / "bootstrap_95ci_all_models.csv",
    TABLE_DIR / "paired_bootstrap_calstack_vs_comparators.csv",
    TABLE_DIR / "mcnemar_calstack_vs_comparators.csv",
    TABLE_DIR / "Table6_per_class_recall.csv",
    TABLE_DIR / "Table_calibration.csv",
    TABLE_DIR / "Table_selective_prediction.csv",
    TABLE_DIR / "Table_source_importance_loso.csv",
    TABLE_DIR / "Table7_configuration_cost.csv",
    LATEX_DIR / "Table4_body.tex", LATEX_DIR / "Table5_body.tex",
    LATEX_DIR / "Table6_body.tex", LATEX_DIR / "Table7_body.tex",
    FIG_DIR / "Fig3_reliability.png", FIG_DIR / "Fig4_risk_coverage.png",
    FIG_DIR / "Fig5_complementarity.png", FIG_DIR / "Fig6_source_contribution.png",
    ROOT / "MANUSCRIPT_VALUES.json", ROOT / "MANUSCRIPT_SNIPPETS.txt",
]
missing = [str(p) for p in REQUIRED if not p.exists()]
if missing:
    raise FileNotFoundError("Missing J9 outputs:\n" + "\n".join(missing))

print("=" * 100)
print("J9 COMPLETE")
print("=" * 100)
print("ZIP :", ZIP_PATH, f"({ZIP_PATH.stat().st_size/1e6:.1f} MB)")
print("Tables :", len(list(TABLE_DIR.glob('*.csv'))))
print("LaTeX  :", len(list(LATEX_DIR.glob('*.tex'))))
print("Figures:", len(list(FIG_DIR.glob('*.png'))), "main +",
      len(list(ESM_DIR.glob('*.png'))), "ESM")
print("\nNext: open MANUSCRIPT_SNIPPETS.txt and replace every \\TBD{} in the .tex,")
print("      then paste latex/Table*_body.tex into the matching tabular environments.")

J9 COMPLETE
ZIP : /kaggle/working/J9_MBEC_RESULTS_PACKAGE.zip (9.8 MB)
Tables : 28
LaTeX  : 4
Figures: 6 main + 8 ESM

Next: open MANUSCRIPT_SNIPPETS.txt and replace every \TBD{} in the .tex,
      then paste latex/Table*_body.tex into the matching tabular environments.
